# AMML ranking competition 2026


In [1]:
from google.colab import files
from pathlib import Path

uploaded = files.upload()
data_dir = Path("data")
data_dir.mkdir(exist_ok=True)

for name, content in uploaded.items():
    target = data_dir / Path(name).name
    target.write_bytes(content)
    print("Загружен:", target)

required = ("train.csv", "test.csv", "sample_submission.csv")
missing = [name for name in required if not (data_dir / name).is_file()]
assert not missing, f"Не найдены файлы: {missing}. Выбери все три CSV."
print("Данные готовы:", data_dir.resolve())


Saving sample_submission.csv to sample_submission.csv
Saving test.csv to test.csv
Saving train.csv to train.csv
Загружен: data/sample_submission.csv
Загружен: data/test.csv
Загружен: data/train.csv
Данные готовы: /content/data


In [3]:
import os, sys, platform, subprocess, time, base64, gzip, glob, json
import numpy as np, pandas as pd
print(platform.platform(), "| python", platform.python_version())
try:
    import torch; NGPU = torch.cuda.device_count(); print("torch", torch.__version__, "| GPU:", [torch.cuda.get_device_name(i) for i in range(NGPU)])
except ImportError:
    NGPU = 0
DATA = next((d for d in ["data", "/kaggle/input/competitions/amml-ranking-competition-2026", "/kaggle/input/amml-ranking-competition-2026"]
             if os.path.exists(os.path.join(d, "train.csv"))), "data")
ART = "artifacts"; os.makedirs(ART, exist_ok=True); os.makedirs("src", exist_ok=True); os.makedirs("jina", exist_ok=True)
PY = sys.executable

RUN = dict(features=True, ce=True, zs_bge=True, zs_qwen=True, stack=True, lora=True,
           jina_v2_zs=True, jina_v2_ft=True, jina_v3_zs=True, jina_v3_ft=True)
print("DATA =", DATA, "| ART =", ART)

def sh(args, env=None, log=None):

    print("$", " ".join(map(str, args)), flush=True)
    e = dict(os.environ, PYTHONUNBUFFERED="1", TOKENIZERS_PARALLELISM="false", **(env or {}))
    p = subprocess.Popen(list(map(str, args)), env=e, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    lf = open(log, "w") if log else None
    for line in p.stdout:
        if lf: lf.write(line); lf.flush()
        if not any(w in line for w in ("Warning", "warn(", "Batches", "Loading weights")): print(line, end="")
    rc = p.wait(); assert rc == 0, f"код возврата {rc}"

def par(cmds, logs):

    ps = []
    for k, (c, lg) in enumerate(zip(cmds, logs)):
        print(f"[GPU{k}] $", " ".join(map(str, c)), flush=True)
        e = dict(os.environ, PYTHONUNBUFFERED="1", TOKENIZERS_PARALLELISM="false", CUDA_VISIBLE_DEVICES=str(k % max(NGPU, 1)))
        ps.append(subprocess.Popen(list(map(str, c)), env=e, stdout=open(lg, "w"), stderr=subprocess.STDOUT))
    while any(p.poll() is None for p in ps):
        time.sleep(120)
        for lg in logs:
            t = open(lg).read().strip().splitlines(); print(f"[{lg}]", t[-1][:200] if t else "")
    for p, lg in zip(ps, logs):
        if p.returncode != 0: print(open(lg).read()[-3000:]); raise RuntimeError(f"{lg}: код {p.returncode}")

def done(*paths): return all(os.path.exists(p) for p in paths)

Linux-6.6.159+-x86_64-with-glibc2.39 | python 3.13.16
torch 2.14.1+cu130 | GPU: ['Tesla T4']
DATA = data | ART = artifacts


## 0. Код решения


In [4]:
%%writefile src/__init__.py
pass


Writing src/__init__.py


In [5]:
%%writefile src/compat.py







import importlib.metadata as md
import sys


def hide_incompatible_torchao(min_version="0.16.0", verbose=True):
    try:
        v = md.version("torchao")
    except md.PackageNotFoundError:
        return False
    from packaging.version import parse
    if parse(v) >= parse(min_version):
        return False
    if "torchao" not in sys.modules or sys.modules["torchao"] is not None:
        sys.modules["torchao"] = None
        if verbose:
            print(f"[compat] torchao {v} < {min_version} несовместим с peft: скрыт для этого процесса", flush=True)
    return True


def force_sdpa_no_gqa(verbose=True):



    try:
        import transformers.integrations.sdpa_attention as sa
    except Exception as e:
        if verbose:
            print(f"[compat] sdpa_attention недоступен ({type(e).__name__}) — GQA-патч не применён", flush=True)
        return False
    if not hasattr(sa, "use_gqa_in_sdpa"):
        if verbose:
            print("[compat] в этой версии transformers нет use_gqa_in_sdpa — патч не нужен/не применён", flush=True)
        return False
    sa.use_gqa_in_sdpa = lambda *a, **k: False
    if verbose:
        print("[compat] SDPA: enable_gqa отключён -> memory-efficient ядро вместо math (O(L) память внимания)", flush=True)
    return True


Writing src/compat.py


In [ ]:
%%writefile src/common.py

import html
import os
import re
import unicodedata

import numpy as np
import pandas as pd

_TAG = re.compile(r"</?[a-zA-Z][^<>]{0,200}>")
_CTRL = re.compile(r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f­​-‏⁠﻿̀́]")
_WS = re.compile(r"\s+")
_MOJI = re.compile(r"[РС][\u0080-ӿ]")


def clean_v2() -> bool:


    return os.environ.get("AMML_CLEAN_V2", "0") == "1"


def fix_mojibake(s: str) -> str:

    if len(s) < 4 or len(_MOJI.findall(s)) < 0.25 * len(s):
        return s
    try:
        return s.encode("cp1251").decode("utf-8")
    except Exception:
        if clean_v2():
            return s
        try:
            return s.encode("cp1251", errors="ignore").decode("utf-8", errors="ignore")
        except Exception:
            return s


def clean_text(s) -> str:
    if not isinstance(s, str):
        return ""
    s = fix_mojibake(s)
    for _ in range(3):
        u = html.unescape(s)
        if u == s:
            break
        s = u
    s = _TAG.sub(" ", s)
    s = _CTRL.sub("", s)
    s = unicodedata.normalize("NFKC", s)
    s = _WS.sub(" ", s).strip()
    return s


def clean_title(s) -> str:
    s = clean_text(s)
    if len(s) > 3 and s.isupper():
        if not (clean_v2() and (len(s) <= 6 or any(ch.isdigit() for ch in s))):
            s = s.capitalize()
    return s


def parse_num(x):

    if x is None or (isinstance(x, float) and np.isnan(x)):
        return np.nan
    s = re.sub(r"[\s  ]", "", str(x)).replace(",", ".")
    try:
        return float(s)
    except ValueError:
        return np.nan


def load_data(data_dir="data", clean=True):
    tr = pd.read_csv(os.path.join(data_dir, "train.csv"))
    te = pd.read_csv(os.path.join(data_dir, "test.csv"))
    for df in (tr, te):
        df["article_char_count"] = df["article_char_count"].map(parse_num)
        df.loc[df["article_passage_count"] <= 0, "article_passage_count"] = np.nan
        df["title_missing"] = df["title"].isna().astype(int)
        if clean:
            df["raw_text_len"] = df["text"].str.len()
            df["noise_html"] = df["text"].str.contains(r"<[a-zA-Z/]|&[a-z]+;", regex=True).astype(int)
            df["noise_moji"] = df["text"].map(lambda s: int(fix_mojibake(s) != s))
            df["query"] = df["query"].map(clean_text)
            df["title"] = df["title"].map(clean_title)
            df["text"] = df["text"].map(clean_text)
        else:
            df["title"] = df["title"].fillna("")
    return tr, te


def make_folds(tr: pd.DataFrame, n_folds=5, seed=42) -> np.ndarray:






    from scipy.sparse import coo_matrix
    from scipy.sparse.csgraph import connected_components

    q = tr["query_id"].astype("category").cat.codes.values
    d = tr["document_id"].astype("category").cat.codes.values
    nq = q.max() + 1
    n = nq + d.max() + 1
    g = coo_matrix((np.ones(len(tr)), (q, d + nq)), shape=(n, n))
    _, lab = connected_components(g, directed=False)
    comp = lab[q]
    sizes = pd.Series(comp).value_counts()
    rng = np.random.RandomState(seed)
    order = sizes.sample(frac=1.0, random_state=rng).sort_values(ascending=False, kind="mergesort")
    load = np.zeros(n_folds)
    assign = {}
    for c, s in order.items():
        f = int(np.argmin(load))
        assign[c] = f
        load[f] += s
    return np.array([assign[c] for c in comp])


def build_passage(title: str, text: str) -> str:
    return f"{title}. {text}" if title else text


def get_device(name="auto"):
    import torch
    if name != "auto":
        return torch.device(name)
    if torch.cuda.is_available():
        return torch.device("cuda")
    if getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


def tag_of(model_name: str) -> str:
    return re.sub(r"[^a-zA-Z0-9]+", "-", model_name.split("/")[-1]).strip("-").lower()


Writing src/common.py


In [6]:
%%writefile src/metric.py

import numpy as np
import pandas as pd


def ndcg_at_k(df: pd.DataFrame, pred_col="prediction", rel_col="relevance", k=10, per_query=False):
    d = df[["query_id", "id", pred_col, rel_col]].copy()
    d["id"] = d["id"].astype(str)
    d = d.sort_values(["id"], kind="mergesort").sort_values([pred_col], ascending=False, kind="mergesort")
    d["rank"] = d.groupby("query_id", sort=False).cumcount() + 1
    d["gain"] = np.power(2.0, d[rel_col].astype(float)) - 1.0
    top = d[d["rank"] <= k]
    dcg = (top["gain"] / np.log2(top["rank"] + 1)).groupby(top["query_id"]).sum()
    i = d.sort_values(rel_col, ascending=False, kind="mergesort")
    i["irank"] = i.groupby("query_id", sort=False).cumcount() + 1
    i = i[i["irank"] <= k]
    idcg = (i["gain"] / np.log2(i["irank"] + 1)).groupby(i["query_id"]).sum()
    dcg = dcg.reindex(idcg.index).fillna(0.0)
    nd = (dcg / idcg.where(idcg > 0)).fillna(0.0)
    return nd if per_query else float(nd.mean())


def score(df: pd.DataFrame, preds, k=10):

    t = df[["query_id", "id", "relevance"]].copy()
    t["prediction"] = np.asarray(preds, dtype=float)
    return ndcg_at_k(t, k=k)


def fold_scores(df, preds, folds):
    return [score(df[folds == f], np.asarray(preds)[folds == f]) for f in sorted(np.unique(folds))]


if __name__ == "__main__":
    t = pd.DataFrame({"query_id": ["a"] * 3 + ["b"] * 2 + ["c"], "id": ["3", "1", "2", "5", "4", "6"],
                      "relevance": [1, 0, 1, 0, 1, 0], "prediction": [0.5, 0.5, 0.1, 0.3, 0.3, 1.0]})

    a = (1 / np.log2(3) + 0.5) / (1 + 1 / np.log2(3)); b = 1.0; c = 0.0
    assert abs(ndcg_at_k(t) - (a + b + c) / 3) < 1e-12, ndcg_at_k(t)
    print("metric ok", ndcg_at_k(t))


Writing src/metric.py


In [ ]:
%%writefile src/losses.py

import torch
import torch.nn.functional as F


def _disc(pos, k):
    return torch.where(pos <= k, 1.0 / torch.log2(pos + 1.0), torch.zeros_like(pos))


@torch.no_grad()
def delta_ndcg_pairs(s, y, k=10):

    s, y = s.detach().float(), y.detach().float()
    n = s.numel()
    order = torch.argsort(s, descending=True, stable=True)
    pos = torch.empty(n, dtype=torch.float32, device=s.device)
    pos[order] = torch.arange(1, n + 1, dtype=torch.float32, device=s.device)
    gain = torch.pow(2.0, y) - 1.0
    disc = _disc(pos, k)
    ip = torch.arange(1, n + 1, dtype=torch.float32, device=s.device)
    idcg = (torch.sort(gain, descending=True).values * _disc(ip, k)).sum().clamp_min(1e-12)
    i, j = torch.nonzero(y[:, None] > y[None, :], as_tuple=True)
    w = ((gain[i] - gain[j]) * (disc[i] - disc[j])).abs() / idcg
    return i, j, w


def lambda_loss(s, y, k=10, only_misordered=False):


    i, j, w = delta_ndcg_pairs(s, y, k)
    if len(i) == 0:
        return None
    if only_misordered:
        w = w * (s.detach()[i] <= s.detach()[j]).float()
    return (w * F.softplus(-(s[i] - s[j]))).mean()


def pp_loss(s, y):

    pos, neg = s[y > 0], s[y <= 0]
    if len(pos) == 0 or len(neg) == 0:
        return None
    lse = torch.logsumexp(torch.cat([pos[:, None], neg[None, :].expand(len(pos), -1)], 1), 1)
    return (lse - pos).mean()


def loss_parts(kind):

    parts = {p.strip() for p in kind.split("+") if p.strip()}
    if "lambdarank" in parts:
        parts = (parts - {"lambdarank"}) | {"lambda"}
    unknown = parts - {"bce", "list", "pp", "lambda", "ranknet"}
    assert not unknown, f"неизвестные компоненты лосса: {unknown}"
    return parts


Writing src/losses.py


In [7]:
%%writefile src/features.py

import argparse
import math
import os
import re
from collections import Counter
from functools import lru_cache

import numpy as np
import pandas as pd

from .common import build_passage, get_device, load_data, make_folds, tag_of
from .metric import fold_scores, score

TOK = re.compile(r"[a-zа-яё0-9]+")
STOP = set("и в во не что он на я с со как а то все она так его но да ты к у же вы за бы по только ее мне было вот от меня еще нет о из ему "
           "теперь когда даже ну вдруг ли если уже или ни быть был него до вас нибудь опять уж вам ведь там потом себя ничего ей может они тут где "
           "есть надо ней для мы тебя их чем была сам чтоб без будто чего раз тоже себе под будет ж тогда кто этот того потому этого какой совсем "
           "ним здесь этом один почти мой тем чтобы нее сейчас были куда зачем всех никогда можно при наконец два об другой хоть после над больше "
           "тот через эти нас про всего них какая много разве три эту моя впрочем хорошо свою этой перед иногда лучше чуть том нельзя такой им более "
           "всегда конечно всю между это который сколько каком какие каких какое".split())

_morph = None


@lru_cache(maxsize=500000)
def lemma(w: str) -> str:
    global _morph
    if _morph is None:
        import pymorphy3
        _morph = pymorphy3.MorphAnalyzer()
    return _morph.parse(w)[0].normal_form.replace("ё", "е")


def tokenize(s: str):
    return TOK.findall(s.lower().replace("ё", "е"))


def lemmas(s: str):
    return [lemma(w) for w in tokenize(s)]


class BM25:
    def __init__(self, docs, k1=1.2, b=0.75):
        self.k1, self.b = k1, b
        self.n = len(docs)
        self.avgdl = sum(len(d) for d in docs) / max(1, self.n)
        df = Counter()
        for d in docs:
            df.update(set(d))
        self.idf = {t: math.log(1 + (self.n - c + 0.5) / (c + 0.5)) for t, c in df.items()}
        self.max_idf = math.log(1 + (self.n + 0.5) / 0.5)

    def score(self, q, d_counts, dl):
        s = 0.0
        for t in q:
            f = d_counts.get(t, 0)
            if f:
                s += self.idf.get(t, self.max_idf) * f * (self.k1 + 1) / (f + self.k1 * (1 - self.b + self.b * dl / self.avgdl))
        return s


def group_feats(df, col):
    g = df.groupby("query_id")[col]
    df[col + "_rank"] = g.rank(ascending=False, method="average")
    mu, sd = g.transform("mean"), g.transform("std").fillna(0)
    df[col + "_z"] = (df[col] - mu) / (sd + 1e-6)
    mx = g.transform("max")
    df[col + "_dmax"] = df[col] - mx
    return df


def embed_cos(model_name, queries, passages, device, batch_size=64, max_len=384):

    import torch
    from sentence_transformers import SentenceTransformer
    m = SentenceTransformer(model_name, device=str(device))
    m.max_seq_length = max_len
    if device.type == "cuda":
        m.half()
    e5 = "e5" in model_name.lower()
    uq = pd.unique(queries)
    up = pd.unique(passages)
    qe = m.encode([("query: " + x) if e5 else x for x in uq], batch_size=batch_size, normalize_embeddings=True, show_progress_bar=False)
    pe = m.encode([("passage: " + x) if e5 else x for x in up], batch_size=batch_size, normalize_embeddings=True, show_progress_bar=True)
    qi = pd.Series(np.arange(len(uq)), index=uq).loc[queries].values
    pi = pd.Series(np.arange(len(up)), index=up).loc[passages].values
    del m
    if device.type == "mps":
        torch.mps.empty_cache()
    return np.einsum("ij,ij->i", qe[qi].astype(np.float32), pe[pi].astype(np.float32))


def build_features(tr, te):
    from sklearn.feature_extraction.text import TfidfVectorizer
    df = pd.concat([tr.assign(_split="train"), te.assign(_split="test")], ignore_index=True)
    print("лемматизация...")
    q_lem = {q: lemmas(q) for q in df["query"].unique()}
    df["passage"] = [build_passage(a, b) for a, b in zip(df["title"], df["text"])]
    docs = df.drop_duplicates("document_id").set_index("document_id")
    text_lem = {i: lemmas(t) for i, t in docs["text"].items()}
    title_lem = {i: lemmas(t) for i, t in docs["title"].items()}
    ids = list(text_lem)
    bm_text = BM25([text_lem[i] for i in ids])
    bm_title = BM25([title_lem[i] for i in ids])
    bm_full = BM25([title_lem[i] + text_lem[i] for i in ids])
    cnt_text = {i: Counter(text_lem[i]) for i in ids}
    cnt_title = {i: Counter(title_lem[i]) for i in ids}

    rows = []
    for q, d, qraw, traw in zip(df["query"], df["document_id"], df["query"].str.lower(), df["text"].str.lower()):
        ql = q_lem[q]
        qc = [t for t in ql if t not in STOP] or ql
        ct, ctt = cnt_text[d], cnt_title[d]
        cf = ct + ctt
        tl, ttl = text_lem[d], title_lem[d]
        idf = [bm_text.idf.get(t, bm_text.max_idf) for t in qc]
        hit_t = [t in ct for t in qc]
        hit_ti = [t in ctt for t in qc]
        tset = set(ttl) - STOP
        qset = set(qc)
        qnums = [t for t in ql if t.isdigit()]

        pos = [i for i, t in enumerate(tl) if t in qset]
        span = (pos[-1] - pos[0] + 1) / max(1, len(tl)) if len(pos) > 1 else 1.0
        bigr_q = set(zip(qc, qc[1:]))
        bigr_t = set(zip(tl, tl[1:]))
        rows.append((
            bm_text.score(qc, ct, len(tl)), bm_title.score(qc, ctt, len(ttl)), bm_full.score(qc, cf, len(tl) + len(ttl)),
            float(np.mean(hit_t)), float(np.mean(hit_ti)),
            float(np.dot(idf, hit_t) / (sum(idf) + 1e-9)), float(np.dot(idf, hit_ti) / (sum(idf) + 1e-9)),
            len(tset & qset) / max(1, len(tset)), len(tset & qset) / max(1, len(tset | qset)),
            len(bigr_q & bigr_t) / max(1, len(bigr_q)), span, len(pos) / max(1, len(tl)),
            len(ql), len(qc), len(tl), len(ttl), len(set(tl)) / max(1, len(tl)),
            sum(t.isdigit() for t in tl) / max(1, len(tl)), int(any(t.isdigit() and len(t) == 4 for t in tl)),
            float(np.mean([t in ct for t in qnums])) if qnums else -1.0,
        ))
    cols = ["bm25_text", "bm25_title", "bm25_full", "q_cov_text", "q_cov_title", "q_idfcov_text", "q_idfcov_title",
            "title_cov_q", "title_q_jacc", "q_bigram_cov", "match_span", "match_density",
            "q_len", "q_len_content", "text_ntok", "title_ntok", "text_uniq_ratio", "text_digit_ratio", "text_has_year", "q_num_cov"]
    df[cols] = pd.DataFrame(rows, index=df.index, columns=cols)

    print("tf-idf...")
    q_str = df["query"].map(lambda q: " ".join(q_lem[q]))
    p_str = df["document_id"].map(lambda d: " ".join(title_lem[d] + text_lem[d]))
    for name, vec, qs, ps in [
        ("tfidf_word", TfidfVectorizer(sublinear_tf=True, min_df=2, ngram_range=(1, 2), token_pattern=r"\S+"), q_str, p_str),
        ("tfidf_char", TfidfVectorizer(sublinear_tf=True, min_df=2, analyzer="char_wb", ngram_range=(2, 4), max_features=300000),
         df["query"].str.lower(), df["passage"].str.lower()),
    ]:
        vec.fit(pd.concat([ps.drop_duplicates(), qs.drop_duplicates()]))
        Q, P = vec.transform(qs), vec.transform(ps)
        df[name] = np.asarray(Q.multiply(P).sum(axis=1)).ravel()


    df["text_len"] = df["text"].str.len()
    df["title_len"] = df["title"].str.len()
    df["src"] = df["ingest_source"].str[1:].astype(int)
    df["pidx_ratio"] = df["passage_index"] / df["article_passage_count"]
    df["is_first"] = (df["passage_index"] == 0).astype(float).where(df["passage_index"].notna())
    df["chars_per_passage"] = df["article_char_count"] / df["article_passage_count"]
    df["text_share_of_article"] = df["text_len"] / df["article_char_count"]

    df["n_cand"] = df.groupby("query_id")["id"].transform("size")
    df["art_cnt_in_q"] = df.groupby(["query_id", "article_id"])["id"].transform("size")
    df["art_share_in_q"] = df["art_cnt_in_q"] / df["n_cand"]
    df["n_art_in_q"] = df.groupby("query_id")["article_id"].transform("nunique")
    df["art_cnt_rank_in_q"] = df.groupby("query_id")["art_cnt_in_q"].rank(ascending=False, method="min")
    df["pidx_rank_in_art"] = df.groupby(["query_id", "article_id"])["passage_index"].rank(method="average")
    df["pidx_min_in_art"] = df.groupby(["query_id", "article_id"])["passage_index"].transform("min")
    df["title_bm25_art_max"] = df.groupby(["query_id", "article_id"])["bm25_title"].transform("max")

    df["doc_nq"] = df.groupby(["_split", "document_id"])["query_id"].transform("nunique")
    df["art_nq"] = df.groupby("article_id")["query_id"].transform("nunique")
    df["art_ndoc"] = df.groupby("article_id")["document_id"].transform("nunique")
    for c in ["bm25_text", "bm25_title", "bm25_full", "tfidf_word", "tfidf_char", "q_idfcov_text", "text_len", "passage_index"]:
        df = group_feats(df, c)
    return df


META = ["id", "query_id", "document_id", "article_id", "_split", "relevance"]
DROP = ["query", "title", "text", "passage", "ingest_source"]


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", default="data")
    ap.add_argument("--out", default="artifacts")
    ap.add_argument("--folds", type=int, default=5)
    ap.add_argument("--seed", type=int, default=42)
    ap.add_argument("--emb_models", nargs="*", default=[], help="zero-shot эмбеддеры, напр. intfloat/multilingual-e5-base")
    ap.add_argument("--emb_max_len", type=int, default=384)
    ap.add_argument("--emb_bs", type=int, default=64)
    ap.add_argument("--device", default="auto")
    a = ap.parse_args()
    os.makedirs(a.out, exist_ok=True)
    tr, te = load_data(a.data)
    fp = os.path.join(a.out, "folds.npy")
    if os.path.exists(fp) and len(np.load(fp)) == len(tr):
        folds = np.load(fp)
    else:
        folds = make_folds(tr, a.folds, a.seed)
        np.save(fp, folds)
    df = build_features(tr, te)
    for m in a.emb_models:
        col = "emb_" + tag_of(m).replace("-", "_")
        cache = os.path.join(a.out, col + ".npy")
        if os.path.exists(cache):
            df[col] = np.load(cache)
        else:
            print("эмбеддинги", m)
            df[col] = embed_cos(m, df["query"].values, df["passage"].values, get_device(a.device), a.emb_bs, a.emb_max_len)
            np.save(cache, df[col].values)
        df = group_feats(df, col)
    feats = df.drop(columns=DROP)
    ftr = feats[feats["_split"] == "train"].drop(columns="_split").reset_index(drop=True)
    fte = feats[feats["_split"] == "test"].drop(columns=["_split", "relevance"]).reset_index(drop=True)
    assert (ftr["id"].values == tr["id"].values).all() and (fte["id"].values == te["id"].values).all()
    ftr.to_pickle(os.path.join(a.out, "feats_train.pkl"))
    fte.to_pickle(os.path.join(a.out, "feats_test.pkl"))
    print(f"\nпризнаков: {ftr.shape[1] - 5}; CV NDCG@10 одиночных бейзлайнов (среднее по фолдам | по всем запросам):")
    rng = np.random.RandomState(0)
    base = [("random", rng.rand(len(ftr))), ("-passage_index", -ftr["passage_index"].fillna(1e6).values)]
    base += [(c, ftr[c].values) for c in ["bm25_text", "bm25_title", "bm25_full", "tfidf_word", "tfidf_char", "q_idfcov_text"]
             + [c for c in ftr.columns if c.startswith("emb_") and not c.endswith(("_rank", "_z", "_dmax"))]]
    res = []
    for n, p in base:
        fs = fold_scores(ftr, p, folds)
        res.append((n, np.mean(fs), score(ftr, p)))
        print(f"  {n:32s} {np.mean(fs):.4f} ± {np.std(fs):.4f} | {res[-1][2]:.4f}")
    pd.DataFrame(res, columns=["baseline", "cv_mean", "cv_all"]).to_csv(os.path.join(a.out, "baselines.csv"), index=False)


if __name__ == "__main__":
    main()


Writing src/features.py


In [8]:
%%writefile src/train_ce.py


import argparse
import json
import math
import os
import random
import time

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import numpy as np
import torch
import torch.nn.functional as F

from .common import build_passage, get_device, load_data, make_folds, tag_of
from .losses import lambda_loss, loss_parts, pp_loss
from .metric import score


def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)


def bi_profile(name):

    n = name.lower()
    if "e5" in n:
        return "query: ", "passage: ", "mean"
    if "rosberta" in n:
        return "search_query: ", "search_document: ", "cls"
    if "bge" in n:
        return "", "", "cls"
    return "", "", "mean"


def encode(tok, queries, passages, max_len, arch="cross", prefix=("", "")):

    if arch == "cross":
        enc = tok(list(queries), list(passages), truncation="only_second", max_length=max_len,
                  padding=False, return_token_type_ids=False)
        return enc["input_ids"]
    q = tok([prefix[0] + x for x in queries], truncation=True, max_length=64, padding=False)["input_ids"]
    p = tok([prefix[1] + x for x in passages], truncation=True, max_length=max_len, padding=False)["input_ids"]
    return list(zip(q, p))


class BiEncoder(torch.nn.Module):


    def __init__(self, name, pool="mean"):
        super().__init__()
        self.pool = pool
        from transformers import AutoModel
        self.enc = AutoModel.from_pretrained(name)
        self.scale = torch.nn.Parameter(torch.tensor(20.0))
        self.bias = torch.nn.Parameter(torch.tensor(-16.0))

    def emb(self, ids, att):
        h = self.enc(input_ids=ids, attention_mask=att).last_hidden_state.float()
        if self.pool == "cls":
            return F.normalize(h[:, 0], dim=-1)
        m = att.unsqueeze(-1).float()
        return F.normalize((h * m).sum(1) / m.sum(1).clamp(min=1), dim=-1)

    def forward(self, q, p):
        return self.scale * (self.emb(*q) * self.emb(*p)).sum(-1) + self.bias


def forward(model, items, pad_id, device):

    if isinstance(items[0], tuple):
        return model(collate([x[0] for x in items], pad_id, device), collate([x[1] for x in items], pad_id, device))
    ids, att = collate(items, pad_id, device)
    return model(input_ids=ids, attention_mask=att).logits.view(-1)


def n_tok(x):
    return len(x[1]) if isinstance(x, tuple) else len(x)


def collate(ids_list, pad_id, device):
    L = max(len(x) for x in ids_list)
    ids = torch.full((len(ids_list), L), pad_id, dtype=torch.long)
    att = torch.zeros((len(ids_list), L), dtype=torch.long)
    for i, x in enumerate(ids_list):
        ids[i, :len(x)] = torch.tensor(x)
        att[i, :len(x)] = 1
    return ids.to(device), att.to(device)


@torch.no_grad()
def predict(model, input_ids, pad_id, device, bs, amp):
    model.eval()
    order = np.argsort([-n_tok(x) for x in input_ids])
    out = np.zeros(len(input_ids), dtype=np.float32)
    for i in range(0, len(order), bs):
        idx = order[i:i + bs]
        with torch.autocast("cuda", dtype=torch.float16, enabled=amp):
            logits = forward(model, [input_ids[j] for j in idx], pad_id, device)
        out[idx] = logits.float().cpu().numpy()
    return out


def group_loss(logits, labels, gid, kind, lambda_alpha=1.0, lambda_k=10, lambda_misordered=False):
    parts = loss_parts(kind)
    loss = 0.0
    if "bce" in parts:
        loss = loss + F.binary_cross_entropy_with_logits(logits, labels)
    if "list" in parts:
        ll, n = 0.0, 0
        for g in torch.unique(gid):
            m = gid == g
            y = labels[m]
            if y.sum() > 0 and m.sum() > 1:
                s = logits[m]
                ll = ll + (torch.logsumexp(s, 0) - torch.logsumexp(s[y > 0], 0))
                n += 1
        if n:
            loss = loss + ll / n
    for name, fn, w in (("pp", pp_loss, 1.0),
                        ("lambda", lambda s, y: lambda_loss(s, y, lambda_k, lambda_misordered), lambda_alpha)):
        if name not in parts:
            continue
        vals = [v for v in (fn(logits[gid == g], labels[gid == g]) for g in torch.unique(gid)) if v is not None]
        if vals:
            loss = loss + w * torch.stack(vals).mean()
    return loss


def train_fold(a, tok, tr_ids, tr_y, tr_gid, device, amp, log_prefix, eval_fn=None):
    from transformers import AutoModelForSequenceClassification, get_linear_schedule_with_warmup
    if a.arch == "bi":
        model = BiEncoder(a.model, bi_profile(a.model)[2])
    else:
        model = AutoModelForSequenceClassification.from_pretrained(a.model, num_labels=1)
    if a.grad_ckpt:
        (model.enc if a.arch == "bi" else model).gradient_checkpointing_enable()
    if a.freeze_emb:


        (model.enc if a.arch == "bi" else model).get_input_embeddings().weight.requires_grad_(False)
    model.to(device)
    no_decay = ("bias", "LayerNorm.weight", "layer_norm")
    named = [(n, p) for n, p in model.named_parameters() if p.requires_grad]
    n_all, n_tr = sum(p.numel() for p in model.parameters()), sum(p.numel() for _, p in named)

    params = [
        {"params": [p for n, p in named if not any(k in n for k in no_decay)], "weight_decay": a.wd},
        {"params": [p for n, p in named if any(k in n for k in no_decay)], "weight_decay": 0.0},
    ]
    opt = None
    if a.adam8bit:
        try:
            import bitsandbytes as bnb
            opt = bnb.optim.AdamW8bit(params, lr=a.lr)
        except Exception as e:
            print(f"{log_prefix} bitsandbytes недоступен ({type(e).__name__}), обычный AdamW", flush=True)
    est = (n_all * 4 + n_tr * 4 + n_tr * (2 if opt is not None else 8)) / 2**30
    print(f"{log_prefix} параметров {n_all / 1e6:.0f}M, обучаемых {n_tr / 1e6:.0f}M; веса+град+Adam ≈ {est:.1f} ГБ (+ активации)", flush=True)
    if opt is None:
        opt = torch.optim.AdamW(params, lr=a.lr)
    groups = {}
    for i, g in enumerate(tr_gid):
        groups.setdefault(g, []).append(i)
    groups = list(groups.values())
    steps_per_epoch = math.ceil(len(groups) / a.q_per_batch / a.grad_accum)
    total = steps_per_epoch * a.epochs
    sched = get_linear_schedule_with_warmup(opt, int(a.warmup * total), total)
    scaler = torch.amp.GradScaler("cuda", enabled=amp)
    pad_id = tok.pad_token_id
    rng = random.Random(a.seed)
    t0 = time.time()
    n_oom = 0
    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats()
    for ep in range(a.epochs):
        model.train()
        rng.shuffle(groups)
        run, nb = 0.0, 0
        opt.zero_grad(set_to_none=True)
        batches = [groups[i:i + a.q_per_batch] for i in range(0, len(groups), a.q_per_batch)]
        for bi, b in enumerate(batches):
            if a.dry_run and bi >= a.dry_run:
                break
            idx = [i for g in b for i in g]
            if a.max_docs_step and len(idx) > a.max_docs_step:

                pos = [i for i in idx if tr_y[i] > 0][:a.max_docs_step - 1]
                neg = [i for i in idx if tr_y[i] <= 0]
                rng.shuffle(neg)
                keep_set = set(pos + neg[:a.max_docs_step - len(pos)])
                keep_pos = [k for k, i in enumerate(idx) if i in keep_set]
                gid_full = [k for k, g in enumerate(b) for _ in g]
                idx = [idx[k] for k in keep_pos]
                gid_list = [gid_full[k] for k in keep_pos]
            else:
                gid_list = [k for k, g in enumerate(b) for _ in g]
            y = torch.tensor(tr_y[idx], dtype=torch.float32, device=device)
            gid = torch.tensor(gid_list, device=device)
            try:
                with torch.autocast("cuda", dtype=torch.float16, enabled=amp):
                    logits = forward(model, [tr_ids[i] for i in idx], pad_id, device)
                loss = group_loss(logits.float(), y, gid, a.loss, a.lambda_alpha, a.lambda_k, a.lambda_misordered) / a.grad_accum
                scaler.scale(loss).backward()
            except torch.OutOfMemoryError:
                n_oom += 1
                del idx
                opt.zero_grad(set_to_none=True)
                torch.cuda.empty_cache()
                print(f"{log_prefix} OOM на батче {bi} (всего {n_oom}), пропуск", flush=True)
                if n_oom > 50:
                    raise
                continue
            if (bi + 1) % a.grad_accum == 0 or bi + 1 == len(batches):
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(opt); scaler.update(); sched.step()
                opt.zero_grad(set_to_none=True)
            run += loss.item() * a.grad_accum; nb += 1
            if a.log_every and (bi + 1) % a.log_every == 0:
                print(f"{log_prefix} ep{ep + 1} {bi + 1}/{len(batches)} loss {run / nb:.4f} {time.time() - t0:.0f}s", flush=True)
        msg = f"{log_prefix} ep{ep + 1} loss {run / max(1, nb):.4f} time {time.time() - t0:.0f}s"
        if device.type == "cuda":
            msg += f" peak_mem {torch.cuda.max_memory_allocated() / 2**30:.1f}GB"
        if a.dry_run:
            print(msg + " (dry run)", flush=True)
            break
        if eval_fn is not None and (a.eval_each_epoch or ep + 1 == a.epochs):
            msg += f" val NDCG@10 {eval_fn(model):.4f}"
        print(msg, flush=True)
    return model


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", default="data")
    ap.add_argument("--out", default="artifacts")
    ap.add_argument("--model", default="cointegrated/rubert-tiny2")
    ap.add_argument("--arch", default="cross", choices=["cross", "bi"], help="cross-encoder или bi-encoder (напр. e5)")
    ap.add_argument("--tag", default=None)
    ap.add_argument("--epochs", type=int, default=3)
    ap.add_argument("--folds", type=int, default=5)
    ap.add_argument("--only_folds", type=int, nargs="*", default=None, help="обучить только эти фолды (для возобновления/параллели)")
    ap.add_argument("--max_len", type=int, default=256)
    ap.add_argument("--q_per_batch", type=int, default=4, help="запросов в батче (~10 пар на запрос)")
    ap.add_argument("--grad_accum", type=int, default=1)
    ap.add_argument("--infer_bs", type=int, default=64)
    ap.add_argument("--lr", type=float, default=2e-5)
    ap.add_argument("--wd", type=float, default=0.01)
    ap.add_argument("--warmup", type=float, default=0.1)
    ap.add_argument("--loss", default="bce+list",
                    choices=["bce", "list", "bce+list", "pp", "pp+bce", "lambdarank", "lambda+bce", "pp+lambda", "pp+lambda+bce"],
                    help="list: multi-positive softmax; pp: softmax по каждому позитиву; lambda(rank): LambdaRank |ΔNDCG@k|-pairwise")
    ap.add_argument("--lambda_alpha", type=float, default=1.0, help="вес LambdaRank-слагаемого (как α в sem11_2)")
    ap.add_argument("--lambda_k", type=int, default=10, help="отсечка NDCG@k в весах LambdaRank")
    ap.add_argument("--lambda_misordered", action="store_true", help="вес только неверно упорядоченным парам")
    ap.add_argument("--seed", type=int, default=42)
    ap.add_argument("--fold_seed", type=int, default=42, help="сид разбиения (общий для всех моделей и стекинга)")
    ap.add_argument("--device", default="auto")
    ap.add_argument("--fp16", action="store_true")
    ap.add_argument("--grad_ckpt", action="store_true", help="gradient checkpointing: активации не хранятся (~-30% скорости)")
    ap.add_argument("--freeze_emb", action="store_true", help="заморозить word_embeddings (экономит ~3 ГБ у XLM-R large)")
    ap.add_argument("--adam8bit", action="store_true", help="bitsandbytes AdamW8bit (если установлен), иначе обычный AdamW")
    ap.add_argument("--max_docs_step", type=int, default=0, help="макс. пар в одном шаге обучения (0 = без ограничения)")
    ap.add_argument("--dry_run", type=int, default=0, help="N шагов обучения на фолде 0 и выход: проверка запуска и пиковой памяти")
    ap.add_argument("--no_clean", action="store_true")
    ap.add_argument("--eval_each_epoch", action="store_true")
    ap.add_argument("--save_model", action="store_true", help="сохранять веса каждого фолда")
    ap.add_argument("--max_queries", type=int, default=0, help="отладка: подвыборка запросов train")
    ap.add_argument("--log_every", type=int, default=0)
    a = ap.parse_args()

    from transformers import AutoTokenizer
    device = get_device(a.device)
    amp = a.fp16 and device.type == "cuda"
    tag = a.tag or f"{'bi-' if a.arch == 'bi' else ''}{tag_of(a.model)}_len{a.max_len}_s{a.seed}"
    out = os.path.join(a.out, f"ce_{tag}")
    os.makedirs(out, exist_ok=True)
    print(f"device={device} amp={amp} tag={tag}", flush=True)

    tr, te = load_data(a.data, clean=not a.no_clean)
    fp = os.path.join(a.out, "folds.npy")
    if os.path.exists(fp) and len(np.load(fp)) == len(tr) and int(np.load(fp).max()) + 1 == a.folds:


        folds = np.load(fp)
        print("folds: загружены из", fp, flush=True)
    else:
        folds = make_folds(tr, a.folds, a.fold_seed)
        np.save(fp, folds)
    if a.max_queries:
        keep = tr["query_id"].isin(tr["query_id"].drop_duplicates().sample(a.max_queries, random_state=0))
    else:
        keep = np.ones(len(tr), dtype=bool)
    keep = np.asarray(keep)
    tok = AutoTokenizer.from_pretrained(a.model)
    e5 = bi_profile(a.model)[:2]
    tr_ids = encode(tok, tr["query"], [build_passage(t, x) for t, x in zip(tr["title"], tr["text"])], a.max_len, a.arch, e5)
    te_ids = encode(tok, te["query"], [build_passage(t, x) for t, x in zip(te["title"], te["text"])], a.max_len, a.arch, e5)
    lens = np.array([n_tok(x) for x in tr_ids])
    print(f"токенов на пару: mean {lens.mean():.0f}, p90 {np.percentile(lens, 90):.0f}, обрезано до max_len: {(lens >= a.max_len).mean():.1%}", flush=True)
    y = tr["relevance"].values.astype(np.float32)
    gid = tr["query_id"].values

    for f in range(a.folds):
        if a.only_folds is not None and f not in a.only_folds:
            continue
        vp, tp = os.path.join(out, f"fold{f}_val.npy"), os.path.join(out, f"fold{f}_test.npy")
        if os.path.exists(vp) and os.path.exists(tp):
            print(f"[fold {f}] уже готов, пропуск"); continue
        set_seed(a.seed + f)
        tri = np.where((folds != f) & keep)[0]
        vai = np.where(folds == f)[0]
        ev = lambda m: score(tr.iloc[vai], predict(m, [tr_ids[i] for i in vai], tok.pad_token_id, device, a.infer_bs, amp))
        if a.dry_run:

            glen = {}
            for i in tri:
                glen[gid[i]] = glen.get(gid[i], 0) + n_tok(tr_ids[i])
            tri = np.array(sorted(tri, key=lambda i: (-glen[gid[i]], gid[i])))[:a.dry_run * a.q_per_batch * 20]
            a_dry = argparse.Namespace(**{**vars(a), "epochs": 1})
            train_fold(a_dry, tok, [tr_ids[i] for i in tri], y[tri], gid[tri], device, amp, f"[dry fold {f}]", None)
            print("dry run ok"); return
        model = train_fold(a, tok, [tr_ids[i] for i in tri], y[tri], gid[tri], device, amp, f"[fold {f}]", ev)
        pv = predict(model, [tr_ids[i] for i in vai], tok.pad_token_id, device, a.infer_bs, amp)
        pt = predict(model, te_ids, tok.pad_token_id, device, a.infer_bs, amp)
        np.save(vp, pv); np.save(tp, pt)
        fs_ = score(tr.iloc[vai], pv)
        print(f"[fold {f}] NDCG@10 = {fs_:.4f}", flush=True)
        json.dump({"tag": tag, "fold": f, "ndcg10": fs_, "n_val": int(len(vai)), "n_train": int(len(tri)), "args": vars(a)},
                  open(os.path.join(out, f"fold{f}_meta.json"), "w"), ensure_ascii=False, indent=1)
        if a.save_model:
            (model.enc if a.arch == "bi" else model).save_pretrained(os.path.join(out, f"model_fold{f}")); tok.save_pretrained(os.path.join(out, f"model_fold{f}"))
        del model
        if device.type == "cuda":
            torch.cuda.empty_cache()
        elif device.type == "mps":
            torch.mps.empty_cache()

    done = [f for f in range(a.folds) if os.path.exists(os.path.join(out, f"fold{f}_test.npy"))]
    if len(done) == a.folds:
        oof = np.zeros(len(tr), dtype=np.float32)
        fs = []
        for f in range(a.folds):
            vai = np.where(folds == f)[0]
            oof[vai] = np.load(os.path.join(out, f"fold{f}_val.npy"))
            fs.append(score(tr.iloc[vai], oof[vai]))
        test = np.mean([np.load(os.path.join(out, f"fold{f}_test.npy")) for f in range(a.folds)], axis=0)
        np.save(os.path.join(out, "oof.npy"), oof); np.save(os.path.join(out, "test.npy"), test)
        meta = {"tag": tag, "cv_mean": float(np.mean(fs)), "cv_std": float(np.std(fs)), "cv_folds": fs,
                "cv_all": score(tr, oof), "args": vars(a)}
        json.dump(meta, open(os.path.join(out, "meta.json"), "w"), ensure_ascii=False, indent=1)
        print(f"CV NDCG@10 [{tag}]: {meta['cv_mean']:.4f} ± {meta['cv_std']:.4f} (по фолдам {np.round(fs, 4).tolist()})")
    else:
        print(f"готовы фолды {done} из {a.folds}; oof/test соберутся, когда будут все")


if __name__ == "__main__":
    main()


Writing src/train_ce.py


In [9]:
%%writefile src/zs_rerank.py

import argparse
import json
import os
import time

import numpy as np
import pandas as pd

from .common import build_passage, clean_v2, get_device, load_data


def select_rows(tr, te, n_q):

    qs = np.random.RandomState(123).choice(tr.query_id.unique(), 400, replace=False)[:n_q]
    return tr.query_id.isin(qs).values, te.query_id.isin(te.query_id.unique()[:n_q]).values


def make_batches(lens, order, tok_budget, max_bs):

    out, cur = [], []
    for j in order:
        if cur and ((len(cur) + 1) * lens[cur[0]] > tok_budget or len(cur) >= max_bs):
            out.append(cur); cur = []
        cur.append(j)
    if cur:
        out.append(cur)
    return out


def run_shard(a, pairs_q, pairs_p, idx_all):
    import torch
    from transformers import AutoModelForSequenceClassification, AutoTokenizer
    dev = get_device(a.device)
    fp16 = a.fp16 and dev.type == "cuda"
    tok = AutoTokenizer.from_pretrained(a.model)
    t0 = time.time()
    enc = tok([pairs_q[i] for i in idx_all], [pairs_p[i] for i in idx_all], truncation="only_second",
              max_length=a.max_len, padding=False, return_token_type_ids=False)["input_ids"]
    lens = np.array([len(x) for x in enc])
    order = np.argsort(-lens, kind="stable")[a.shard::a.n_shards]
    print(f"[shard {a.shard}/{a.n_shards}] токенизация {time.time() - t0:.0f}s; пар {len(order)} из {len(enc)}; "
          f"токенов: mean {lens.mean():.0f}, p99 {np.percentile(lens, 99):.0f}, max {lens.max()}, "
          f"обрезано до {a.max_len}: {(lens >= a.max_len).mean():.2%}", flush=True)
    kw = {}
    try:
        model = AutoModelForSequenceClassification.from_pretrained(a.model, attn_implementation="sdpa")
        kw["attn"] = "sdpa"
    except Exception as e:
        print("sdpa недоступен:", type(e).__name__, e, flush=True)
        model = AutoModelForSequenceClassification.from_pretrained(a.model)
    model = (model.half() if fp16 else model).to(dev).eval()
    pad = tok.pad_token_id
    scores = np.full(len(enc), np.nan, dtype=np.float32)

    def fwd(b):
        L = int(max(lens[j] for j in b))
        ids = torch.full((len(b), L), pad, dtype=torch.long); att = torch.zeros((len(b), L), dtype=torch.long)
        for k, j in enumerate(b):
            ids[k, :lens[j]] = torch.tensor(enc[j]); att[k, :lens[j]] = 1
        with torch.no_grad():
            out = model(input_ids=ids.to(dev), attention_mask=att.to(dev)).logits.view(-1).float().cpu().numpy()
        scores[b] = out

    def safe(b):
        try:
            fwd(b)
        except torch.OutOfMemoryError:
            if len(b) == 1:
                raise
            torch.cuda.empty_cache()
            print(f"  OOM на пакете {len(b)}x{lens[b[0]]} -> делю пополам", flush=True)
            safe(b[:len(b) // 2]); safe(b[len(b) // 2:])

    batches = make_batches(lens, order, a.tok_budget, a.max_bs)
    t0, done = time.time(), 0
    for bi, b in enumerate(batches):
        safe(b)
        done += len(b)
        if (bi + 1) % a.log_every == 0 or bi + 1 == len(batches):
            el = time.time() - t0
            print(f"  [shard {a.shard}] {done}/{len(order)} пар, {el:.0f}s, {done / el:.1f} пар/с, "
                  f"ETA {(len(order) - done) / max(done / el, 1e-9):.0f}s", flush=True)
    assert np.isfinite(scores[order]).all(), "есть нечисловые скоры"
    return scores, order, {**kw, "fp16": fp16, "device": str(dev), "sec": time.time() - t0, "n_pairs": int(len(order)),
                           "tok_mean": float(lens.mean()), "trunc_share": float((lens >= a.max_len).mean())}


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", default="data")
    ap.add_argument("--out", default="artifacts")
    ap.add_argument("--model", default="BAAI/bge-reranker-v2-m3")
    ap.add_argument("--name", default="zs_rr1024")
    ap.add_argument("--max_len", type=int, default=1024)
    ap.add_argument("--fp16", action="store_true", help="fp16 на CUDA (на CPU/MPS игнорируется)")
    ap.add_argument("--tok_budget", type=int, default=32768, help="макс. токенов в пакете (n * max_len пакета)")
    ap.add_argument("--max_bs", type=int, default=128)
    ap.add_argument("--device", default="auto")
    ap.add_argument("--shard", type=int, default=0)
    ap.add_argument("--n_shards", type=int, default=1)
    ap.add_argument("--merge", action="store_true", help="только собрать <name>_part*.npz в итоговые npy")
    ap.add_argument("--dry_queries", type=int, default=0, help=">0: только N запросов train (из 400 ревью) и N test")
    ap.add_argument("--check", default=None, help="pkl/csv с колонками id, zs1024 (review/zs_reranker_sub.pkl) для сверки")
    ap.add_argument("--log_every", type=int, default=50)
    a = ap.parse_args()
    os.makedirs(a.out, exist_ok=True)

    tr, te = load_data(a.data)
    n_tr, n_te = len(tr), len(te)
    q = list(tr["query"]) + list(te["query"])
    p = [build_passage(t, x) for t, x in zip(tr["title"], tr["text"])] + [build_passage(t, x) for t, x in zip(te["title"], te["text"])]
    if a.dry_queries:
        mtr, mte = select_rows(tr, te, a.dry_queries)
        idx_all = np.concatenate([np.where(mtr)[0], n_tr + np.where(mte)[0]])
    else:
        idx_all = np.arange(n_tr + n_te)
    part = os.path.join(a.out, f"{a.name}_part{a.shard}of{a.n_shards}.npz")
    if not a.merge:
        s, order, info = run_shard(a, q, p, idx_all)
        np.savez(part, rows=idx_all[order], scores=s[order], info=json.dumps(info))
        print(f"[shard {a.shard}] сохранено {part}: {info}", flush=True)
        if a.n_shards > 1:
            return
    full = np.full(n_tr + n_te, np.nan, dtype=np.float32)
    infos = []
    for k in range(a.n_shards):
        z = np.load(os.path.join(a.out, f"{a.name}_part{k}of{a.n_shards}.npz"))
        full[z["rows"]] = z["scores"]; infos.append(json.loads(str(z["info"])))
    got = np.isfinite(full)
    assert got[idx_all].all(), f"не хватает {(~got[idx_all]).sum()} строк — не все шарды досчитаны?"
    ptr, pte = os.path.join(a.out, f"{a.name}_train.npy"), os.path.join(a.out, f"{a.name}_test.npy")
    np.save(ptr, full[:n_tr]); np.save(pte, full[n_tr:])
    from .metric import score
    m = got[:n_tr]
    meta = {"model": a.model, "max_len": a.max_len, "name": a.name, "clean_v2": clean_v2(), "dry_queries": a.dry_queries,
            "n_train": n_tr, "n_test": n_te, "rows_done": int(got.sum()), "shards": infos,
            "train_ndcg10": score(tr[m], full[:n_tr][m]) if m.any() else None}
    fp = os.path.join(a.out, "folds.npy")
    if os.path.exists(fp) and not a.dry_queries:
        folds = np.load(fp)
        meta["train_ndcg10_by_fold"] = [score(tr[folds == f], full[:n_tr][folds == f]) for f in range(int(folds.max()) + 1)]
    if a.check and os.path.exists(a.check):
        c = pd.read_pickle(a.check) if a.check.endswith(".pkl") else pd.read_csv(a.check)
        d = pd.DataFrame({"id": tr["id"], "new": full[:n_tr]}).merge(c[["id", "zs1024"]], on="id").dropna()
        if len(d):
            meta["check"] = {"n": int(len(d)), "pearson": float(np.corrcoef(d.new, d.zs1024)[0, 1]),
                             "max_abs_diff": float((d.new - d.zs1024).abs().max()), "mean_abs_diff": float((d.new - d.zs1024).abs().mean())}
    json.dump(meta, open(os.path.join(a.out, f"{a.name}_meta.json"), "w"), ensure_ascii=False, indent=1)
    print(f"сохранено {ptr} {pte}; строк посчитано {int(got.sum())} из {n_tr + n_te}")
    print(json.dumps({k: v for k, v in meta.items() if k != "shards"}, ensure_ascii=False))


if __name__ == "__main__":
    main()


Writing src/zs_rerank.py


In [10]:
%%writefile src/llm_zs_eval.py

import argparse
import json
import os
import time

os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import numpy as np
import pandas as pd
import torch

from .common import build_passage, load_data
from .metric import score
from .train_llm_rerank import DEFAULT_INSTRUCTION, Encoder, forward_scores

DT = {"fp16": torch.float16, "fp32": torch.float32, "bf16": torch.bfloat16}


def load(name, dtype, device_map=None, attn="sdpa"):
    from transformers import AutoModelForCausalLM
    kw = dict(attn_implementation=attn)
    if device_map:
        kw["device_map"] = device_map
    try:
        m = AutoModelForCausalLM.from_pretrained(name, dtype=DT[dtype], **kw)
    except TypeError:
        m = AutoModelForCausalLM.from_pretrained(name, torch_dtype=DT[dtype], **kw)
    m.config.use_cache = False
    if not device_map:
        m = m.to("cuda" if torch.cuda.is_available() else "cpu")
    return m.eval()


def in_device(m):
    return m.get_input_embeddings().weight.device


@torch.no_grad()
def score_rows(m, enc, ids, tok_budget, log="", log_every=20):

    dev = in_device(m)
    lens = np.array([len(x) for x in ids])
    order = np.argsort(-lens, kind="stable")
    out = np.full(len(ids), np.nan, dtype=np.float32)

    def run(b):
        try:
            out[b] = forward_scores(m, enc, [ids[j] for j in b], dev, None).cpu().numpy()
        except torch.OutOfMemoryError:
            if len(b) == 1:
                raise
            torch.cuda.empty_cache(); run(b[:len(b) // 2]); run(b[len(b) // 2:])

    i, nb, t0 = 0, 0, time.time()
    while i < len(order):
        n = max(1, min(256, tok_budget // int(lens[order[i]])))
        run(order[i:i + n]); i = min(i + n, len(order)); nb += 1
        if log and (nb % log_every == 0 or i >= len(order)):
            el = time.time() - t0
            print(f"{log} {i}/{len(order)} пар, {el:.0f}s, {i / el:.1f} пар/с, ETA {(len(order) - i) / max(i / el, 1e-9):.0f}s, "
                  f"нечисловых пока {int((~np.isfinite(out[order[:i]])).sum())}", flush=True)
    return out


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", default="data")
    ap.add_argument("--out", default="artifacts")
    ap.add_argument("--model", default="Qwen/Qwen3-Reranker-4B")
    ap.add_argument("--prompt_style", default="qwen3", choices=["qwen3", "bge_gemma"])
    ap.add_argument("--instruction", default=DEFAULT_INSTRUCTION)
    ap.add_argument("--name", default="qwen3_4b")
    ap.add_argument("--max_len", type=int, default=768)
    ap.add_argument("--dtype", default="fp16", choices=["fp16", "fp32", "bf16"])
    ap.add_argument("--fold", type=int, default=0)
    ap.add_argument("--tok_budget", type=int, default=16384)
    ap.add_argument("--shard", type=int, default=0)
    ap.add_argument("--n_shards", type=int, default=1)
    ap.add_argument("--device_map", default=None, help="'auto': слои модели по всем GPU (для 8B на 2xT4), без шардов")
    ap.add_argument("--merge", action="store_true")
    ap.add_argument("--calib", type=int, default=0, help="при --merge: перепроверить N случайных пар в fp32 (device_map auto)")
    ap.add_argument("--dry_queries", type=int, default=0, help=">0: только первые N запросов фолда")
    ap.add_argument("--inject_nan", type=int, default=0, help="отладка: испортить N скоров шарда, проверить fp32-перепроверку")
    a = ap.parse_args()
    os.makedirs(a.out, exist_ok=True)
    from transformers import AutoTokenizer
    tr, _ = load_data(a.data)
    folds = np.load(os.path.join(a.out, "folds.npy"))
    assert len(folds) == len(tr), "folds.npy не соответствует train.csv"
    rows = np.where(folds == a.fold)[0]
    if a.dry_queries:
        rows = rows[np.isin(tr.query_id.values[rows], list(dict.fromkeys(tr.query_id.values[rows]))[:a.dry_queries])]
    tok = AutoTokenizer.from_pretrained(a.model)
    if tok.pad_token_id is None:
        tok.pad_token = tok.eos_token
    enc = Encoder(tok, a.prompt_style, a.max_len, a.instruction)
    sub = tr.iloc[rows]
    ids = enc(list(sub["query"]), [build_passage(t, x) for t, x in zip(sub["title"], sub["text"])])
    lens = np.array([len(x) for x in ids])
    part = lambda k: os.path.join(a.out, f"{a.name}_part{k}of{a.n_shards}.npz")

    if not a.merge:
        order = np.argsort(-lens, kind="stable")[a.shard::a.n_shards]
        print(f"[{a.name} shard {a.shard}/{a.n_shards}] {a.model} {a.dtype}: пар {len(order)} из {len(ids)}; токенов mean {lens.mean():.0f}, "
              f"max {lens.max()}, на пределе {a.max_len}: {(lens >= a.max_len).mean():.1%}; cuda={torch.cuda.is_available()}", flush=True)
        t0 = time.time(); m = load(a.model, a.dtype, a.device_map); tl = time.time() - t0
        if torch.cuda.is_available():
            print(f"  веса загружены за {tl:.0f}s, память GPU {torch.cuda.memory_allocated() / 2**30:.1f} ГБ", flush=True)
        t0 = time.time()
        s = score_rows(m, enc, [ids[j] for j in order], a.tok_budget, log=f"  [{a.name} shard {a.shard}]")
        if a.inject_nan:
            s[:a.inject_nan] = np.nan
        info = {"sec": time.time() - t0, "load_sec": tl, "n": int(len(order)), "nonfinite": int((~np.isfinite(s)).sum()),
                "peak_gb": torch.cuda.max_memory_allocated() / 2**30 if torch.cuda.is_available() else None}
        np.savez(part(a.shard), idx=order, s=s, info=json.dumps(info))
        print(f"[{a.name} shard {a.shard}] готово: {info}", flush=True)
        if a.n_shards > 1:
            return
    full = np.full(len(ids), np.nan, dtype=np.float32); infos = []
    for k in range(a.n_shards):
        z = np.load(part(k)); full[z["idx"]] = z["s"]; infos.append(json.loads(str(z["info"])))
    bad = np.where(~np.isfinite(full))[0]
    meta = {"model": a.model, "dtype": a.dtype, "max_len": a.max_len, "fold": a.fold, "n_pairs": int(len(ids)),
            "tok_mean": float(lens.mean()), "shards": infos, "nonfinite_first_pass": int(len(bad))}
    rng = np.random.RandomState(0)
    cal = rng.choice(len(ids), min(a.calib, len(ids)), replace=False) if a.calib else np.array([], int)
    need = np.union1d(bad, cal).astype(int)
    if len(need):
        print(f"[{a.name}] fp32-перепроверка: нечисловых {len(bad)}, калибровочных {len(cal)} -> {len(need)} пар "
              f"(device_map={'auto' if torch.cuda.is_available() else 'cpu'})", flush=True)
        t0 = time.time()
        try:
            m32 = load(a.model, "fp32", "auto" if torch.cuda.is_available() else None)
            s32 = score_rows(m32, enc, [ids[j] for j in need], max(2048, a.tok_budget // 2))
            del m32
        except Exception as e:
            print(f"[{a.name}] !!! fp32-перепроверка не удалась: {type(e).__name__}: {str(e)[:300]}", flush=True)
            meta["fp32_failed"] = f"{type(e).__name__}: {str(e)[:300]}"
            s32 = np.full(len(need), np.nan, dtype=np.float32); cal = np.array([], int)
        pos = {j: k for k, j in enumerate(need)}
        if len(cal):
            c16, c32 = full[cal], s32[[pos[j] for j in cal]]
            ok = np.isfinite(c16)
            meta["calib"] = {"n": int(len(cal)), "finite16": int(ok.sum()),
                             "pearson": float(np.corrcoef(c16[ok], c32[ok])[0, 1]) if ok.sum() > 2 else None,
                             "max_abs_diff": float(np.abs(c16[ok] - c32[ok]).max()) if ok.any() else None,
                             "mean_abs_diff": float(np.abs(c16[ok] - c32[ok]).mean()) if ok.any() else None}
        if len(bad):
            full[bad] = s32[[pos[j] for j in bad]]
        meta["fp32_sec"] = time.time() - t0
    meta["nonfinite_final"] = int((~np.isfinite(full)).sum())
    if meta["nonfinite_final"]:
        print(f"[{a.name}] !!! {meta['nonfinite_final']} нечисловых скоров остались -> минимум запроса - 1 (NDCG занижен)", flush=True)
        s = pd.Series(full); qmin = s.groupby(sub.query_id.values).transform("min").fillna(0).values
        full = np.where(np.isfinite(full), full, qmin - 1).astype(np.float32)
    meta["ndcg10"] = score(sub, full)
    np.save(os.path.join(a.out, f"{a.name}_fold{a.fold}_val.npy"), full)
    np.save(os.path.join(a.out, f"{a.name}_fold{a.fold}_rows.npy"), rows)
    json.dump(meta, open(os.path.join(a.out, f"{a.name}_meta.json"), "w"), ensure_ascii=False, indent=1)
    print(f"[{a.name}] NDCG@10 фолда {a.fold} (zero-shot) = {meta['ndcg10']:.4f}; NaN/inf в первом проходе: {len(bad)}; "
          f"калибровка fp16 vs fp32: {meta.get('calib')}", flush=True)


if __name__ == "__main__":
    main()


Writing src/llm_zs_eval.py


In [11]:
%%writefile src/llm_zs_all.py


import argparse
import json
import os
import time

os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import numpy as np
import pandas as pd
import torch

from .common import build_passage, load_data
from .llm_zs_eval import load, score_rows
from .metric import score
from .train_llm_rerank import DEFAULT_INSTRUCTION, Encoder


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", default="data")
    ap.add_argument("--out", default="artifacts")
    ap.add_argument("--model", default="Qwen/Qwen3-Reranker-4B")
    ap.add_argument("--instruction", default=DEFAULT_INSTRUCTION)
    ap.add_argument("--name", default="qwen4b")
    ap.add_argument("--max_len", type=int, default=768)
    ap.add_argument("--dtype", default="fp16", choices=["fp16", "fp32", "bf16"])
    ap.add_argument("--tok_budget", type=int, default=16384)
    ap.add_argument("--chunk", type=int, default=2000, help="пар в одном чекпоинте шарда")
    ap.add_argument("--shard", type=int, default=0)
    ap.add_argument("--n_shards", type=int, default=1)
    ap.add_argument("--merge", action="store_true")
    ap.add_argument("--dry_queries", type=int, default=0, help=">0: только первые N запросов train и N запросов test")
    ap.add_argument("--stop_after_chunks", type=int, default=0, help="отладка: остановиться после N новых кусков (имитация обрыва)")
    ap.add_argument("--inject_nan", type=int, default=0, help="отладка: испортить N скоров первого куска")
    a = ap.parse_args()
    os.makedirs(a.out, exist_ok=True)
    from transformers import AutoTokenizer
    tr, te = load_data(a.data)
    n_tr, n_te = len(tr), len(te)
    df = pd.concat([tr[["query_id", "query", "title", "text"]], te[["query_id", "query", "title", "text"]]], ignore_index=True)
    rows = np.arange(n_tr + n_te)
    if a.dry_queries:
        qs = list(dict.fromkeys(tr.query_id))[:a.dry_queries] + list(dict.fromkeys(te.query_id))[:a.dry_queries]
        rows = rows[df.query_id.isin(qs).values]
    tok = AutoTokenizer.from_pretrained(a.model)
    if tok.pad_token_id is None:
        tok.pad_token = tok.eos_token
    enc = Encoder(tok, "qwen3", a.max_len, a.instruction)
    t0 = time.time()
    sub = df.iloc[rows]
    ids = enc(list(sub["query"]), [build_passage(t, x) for t, x in zip(sub["title"], sub["text"])])
    lens = np.array([len(x) for x in ids])
    print(f"[{a.name}] токенизация {time.time() - t0:.0f}s: пар {len(ids)} (train {int((rows < n_tr).sum())}, test {int((rows >= n_tr).sum())}), "
          f"токенов mean {lens.mean():.0f}, на пределе {a.max_len}: {(lens >= a.max_len).mean():.1%}", flush=True)
    cfile = lambda k, c: os.path.join(a.out, f"{a.name}_s{k}of{a.n_shards}_c{c:03d}.npz")
    shard_order = lambda k: np.argsort(-lens, kind="stable")[k::a.n_shards]

    if not a.merge:
        order = shard_order(a.shard)
        chunks = [order[i:i + a.chunk] for i in range(0, len(order), a.chunk)]
        todo = [c for c in range(len(chunks)) if not os.path.exists(cfile(a.shard, c))]
        tot_tok = lens[order].sum(); done_tok = tot_tok - sum(lens[chunks[c]].sum() for c in todo)
        print(f"[{a.name} shard {a.shard}/{a.n_shards}] пар {len(order)}, кусков {len(chunks)}, уже готово {len(chunks) - len(todo)} "
              f"({done_tok / tot_tok:.0%} токенов)", flush=True)
        if not todo:
            return
        t0 = time.time(); m = load(a.model, a.dtype); print(f"  веса загружены за {time.time() - t0:.0f}s", flush=True)
        t0, new_tok, n_new = time.time(), 0, 0
        for c in todo:
            idx = chunks[c]
            s = score_rows(m, enc, [ids[j] for j in idx], a.tok_budget)
            if a.inject_nan and c == todo[0]:
                s[:a.inject_nan] = np.nan
            tmp = cfile(a.shard, c) + ".tmp.npz"
            np.savez(tmp, idx=idx, s=s, rows=rows[idx]); os.replace(tmp, cfile(a.shard, c))
            new_tok += lens[idx].sum(); done_tok += lens[idx].sum(); n_new += 1
            el = time.time() - t0
            print(f"  [{a.name} shard {a.shard}] кусок {c + 1}/{len(chunks)}: {done_tok / tot_tok:.1%} токенов, {el:.0f}s, "
                  f"{new_tok / el:.0f} ток/с, ETA {(tot_tok - done_tok) / max(new_tok / el, 1e-9):.0f}s, "
                  f"NaN/inf в куске {int((~np.isfinite(s)).sum())}", flush=True)
            if a.stop_after_chunks and n_new >= a.stop_after_chunks:
                print("  (остановка по --stop_after_chunks)", flush=True); return
        return

    full = np.full(len(ids), np.nan, dtype=np.float32)
    for k in range(a.n_shards):
        order = shard_order(k)
        for c in range((len(order) + a.chunk - 1) // a.chunk):
            assert os.path.exists(cfile(k, c)), f"нет куска {cfile(k, c)}: шард {k} не досчитан"
            z = np.load(cfile(k, c))
            assert (z["rows"] == rows[z["idx"]]).all(), "кусок от другой выборки строк/версии данных"
            full[z["idx"]] = z["s"]
    bad = np.where(~np.isfinite(full))[0]
    meta = {"model": a.model, "dtype": a.dtype, "max_len": a.max_len, "n_pairs": int(len(ids)), "tok_mean": float(lens.mean()),
            "nonfinite_first_pass": int(len(bad)), "dry_queries": a.dry_queries}
    print(f"[{a.name}] сборка: NaN/inf в первом проходе {len(bad)} из {len(ids)}", flush=True)
    if len(bad):
        t0 = time.time()
        try:
            m32 = load(a.model, "fp32", "auto" if torch.cuda.is_available() else None)
            full[bad] = score_rows(m32, enc, [ids[j] for j in bad], max(2048, a.tok_budget // 2), log=f"  [{a.name} fp32]")
            del m32
        except Exception as e:
            meta["fp32_failed"] = f"{type(e).__name__}: {str(e)[:300]}"
            print(f"[{a.name}] !!! fp32-пересчёт не удался: {meta['fp32_failed']}", flush=True)
        meta["fp32_sec"] = time.time() - t0
        left = ~np.isfinite(full)
        if left.any():
            qmin = pd.Series(full).groupby(sub.query_id.values).transform("min").fillna(0).values
            full = np.where(left, qmin - 1, full).astype(np.float32)
        meta["nonfinite_final_replaced"] = int(left.sum())
    out_tr = np.full(n_tr, np.nan, dtype=np.float32); out_te = np.full(n_te, np.nan, dtype=np.float32)
    is_tr = rows < n_tr
    out_tr[rows[is_tr]] = full[is_tr]; out_te[rows[~is_tr] - n_tr] = full[~is_tr]
    np.save(os.path.join(a.out, f"{a.name}_train.npy"), out_tr); np.save(os.path.join(a.out, f"{a.name}_test.npy"), out_te)
    m = np.isfinite(out_tr)
    meta["train_ndcg10"] = score(tr[m], out_tr[m])
    fp = os.path.join(a.out, "folds.npy")
    if os.path.exists(fp):
        folds = np.load(fp)
        meta["train_ndcg10_by_fold"] = {int(f): score(tr[(folds == f) & m], out_tr[(folds == f) & m])
                                        for f in range(int(folds.max()) + 1) if ((folds == f) & m).any()}
    meta["rows_train"], meta["rows_test"] = int(m.sum()), int(np.isfinite(out_te).sum())
    json.dump(meta, open(os.path.join(a.out, f"{a.name}_meta.json"), "w"), ensure_ascii=False, indent=1)
    print(f"[{a.name}] сохранено {a.name}_train.npy ({meta['rows_train']}/{n_tr}), {a.name}_test.npy ({meta['rows_test']}/{n_te})")
    print(f"[{a.name}] zero-shot NDCG@10 train: {meta['train_ndcg10']:.4f}; по фолдам "
          + ", ".join(f"{f}: {v:.4f}" for f, v in meta.get("train_ndcg10_by_fold", {}).items()), flush=True)


if __name__ == "__main__":
    main()


Writing src/llm_zs_all.py


In [12]:
%%writefile src/train_llm_rerank.py


import argparse
import json
import math
import os
import random
import sys
import time

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import numpy as np
import torch
import torch.nn.functional as F

from .common import build_passage, clean_v2, load_data
from .losses import lambda_loss, loss_parts, pp_loss
from .metric import score

QWEN3_PREFIX = ("<|im_start|>system\nJudge whether the Document meets the requirements based on the Query and the Instruct "
                "provided. Note that the answer can only be \"yes\" or \"no\".<|im_end|>\n<|im_start|>user\n")
QWEN3_SUFFIX = "<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n"
GEMMA_PROMPT = ("Given a query A and a passage B, determine whether the passage contains an answer to the query by providing "
                "a prediction of either 'Yes' or 'No'.")
DEFAULT_INSTRUCTION = "Given a question in Russian, judge whether the Russian Wikipedia passage contains the answer to the question"


def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)


def style_of(a):
    if a.prompt_style != "auto":
        return a.prompt_style
    n = a.model.lower()
    if "gemma" in n and "bge" in n:
        return "bge_gemma"
    if "qwen3-reranker" in n or "qwen3_reranker" in n:
        return "qwen3"
    raise ValueError(f"не знаю формат промпта для {a.model}: задайте --prompt_style qwen3|bge_gemma")


class Encoder:


    def __init__(self, tok, style, max_len, instruction):
        self.tok, self.style, self.max_len, self.inst = tok, style, max_len, instruction
        enc = lambda s: tok(s, add_special_tokens=False)["input_ids"]
        if style == "qwen3":
            self.prefix, self.suffix = enc(QWEN3_PREFIX), enc(QWEN3_SUFFIX)
            self.yes, self.no = tok.convert_tokens_to_ids("yes"), tok.convert_tokens_to_ids("no")
        else:
            self.sep, self.prompt = enc("\n"), enc(GEMMA_PROMPT)
            self.yes, self.no = enc("Yes")[0], enc("No")[0]
        assert self.yes != tok.unk_token_id and self.no != tok.unk_token_id

    def __call__(self, queries, passages, bs=2048):
        out = []
        for i in range(0, len(queries), bs):
            q, p = queries[i:i + bs], passages[i:i + bs]
            if self.style == "qwen3":
                body = [f"<Instruct>: {self.inst}\n<Query>: {x}\n<Document>: {y}" for x, y in zip(q, p)]
                ids = self.tok(body, add_special_tokens=False, truncation=True,
                               max_length=self.max_len - len(self.prefix) - len(self.suffix))["input_ids"]
                out += [self.prefix + x + self.suffix for x in ids]
            else:
                qi = self.tok([f"A: {x}" for x in q], add_special_tokens=False, truncation=True, max_length=self.max_len * 3 // 4)["input_ids"]
                pi = self.tok([f"B: {y}" for y in p], add_special_tokens=False, truncation=True, max_length=self.max_len)["input_ids"]
                for x, y in zip(qi, pi):
                    first = [self.tok.bos_token_id] + x
                    second = (self.sep + y)[:max(0, self.max_len - len(first))]
                    out.append(first + second + self.sep + self.prompt)
        return out


def load_model(a, device):
    from transformers import AutoModelForCausalLM
    dtype = {"bf16": torch.bfloat16, "fp16": torch.float16, "fp32": torch.float32}[a.dtype]
    kw = dict(attn_implementation=a.attn)
    try:
        model = AutoModelForCausalLM.from_pretrained(a.model, dtype=dtype, **kw)
    except TypeError:
        model = AutoModelForCausalLM.from_pretrained(a.model, torch_dtype=dtype, **kw)
    model.config.use_cache = False
    return model.to(device)


def add_lora(a, model):
    from peft import LoraConfig, get_peft_model
    if a.grad_ckpt:
        model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    cfg = LoraConfig(r=a.lora_r, lora_alpha=a.lora_alpha, lora_dropout=a.lora_dropout, bias="none", task_type="CAUSAL_LM",
                     target_modules=[t.strip() for t in a.lora_targets.split(",") if t.strip()])
    return get_peft_model(model, cfg)


def collate_left(batch, pad_id, device):
    L = max(len(x) for x in batch)
    ids = torch.full((len(batch), L), pad_id, dtype=torch.long)
    att = torch.zeros((len(batch), L), dtype=torch.long)
    for i, x in enumerate(batch):
        ids[i, L - len(x):] = torch.tensor(x)
        att[i, L - len(x):] = 1
    pos = (att.cumsum(-1) - 1).clamp(min=0)
    return ids.to(device), att.to(device), pos.to(device)


_KEEP_ARG = ["logits_to_keep"]


def forward_scores(model, enc, batch, device, amp_dtype):

    ids, att, pos = collate_left(batch, enc.tok.pad_token_id, device)
    with torch.autocast(device.type, dtype=amp_dtype, enabled=amp_dtype is not None):
        try:
            out = model(input_ids=ids, attention_mask=att, position_ids=pos, **{_KEEP_ARG[0]: 1})
        except TypeError:
            _KEEP_ARG[0] = "num_logits_to_keep"
            try:
                out = model(input_ids=ids, attention_mask=att, position_ids=pos, num_logits_to_keep=1)
            except TypeError:
                out = model(input_ids=ids, attention_mask=att, position_ids=pos)
    last = out.logits[:, -1, :].float()
    if enc.style == "qwen3":
        return last[:, enc.yes] - last[:, enc.no]
    return last[:, enc.yes]


@torch.no_grad()
def predict(model, enc, ids_list, device, amp_dtype, tok_budget, max_bs=256, log=None):
    model.eval()
    lens = np.array([len(x) for x in ids_list])
    order = np.argsort(-lens, kind="stable")
    out = np.zeros(len(ids_list), dtype=np.float32)
    i, t0 = 0, time.time()
    while i < len(order):
        n = max(1, min(max_bs, tok_budget // int(lens[order[i]])))
        idx = order[i:i + n]
        out[idx] = forward_scores(model, enc, [ids_list[j] for j in idx], device, amp_dtype).cpu().numpy()
        i += n
    if log:
        print(f"{log} инференс {len(ids_list)} пар за {time.time() - t0:.0f}s", flush=True)
    return out


def group_loss(s, y, gid, a):

    kinds = loss_parts(a.loss)
    total, parts = 0.0, {}
    if "bce" in kinds:
        yt = y * (1 - a.label_smooth) + 0.5 * a.label_smooth
        parts["bce"] = F.binary_cross_entropy_with_logits(s, yt)
        total = total + a.bce_w * parts["bce"]
    rank, lam = 0.0, 0.0; ng = 0
    for g in torch.unique(gid):
        m = gid == g
        sg, yg = s[m] / a.tau, y[m]
        pos, neg = sg[yg > 0], sg[yg <= 0]
        if len(pos) == 0 or len(neg) == 0:
            continue
        ng += 1
        if "pp" in kinds:
            rank = rank + pp_loss(sg, yg)
        if "list" in kinds:
            rank = rank + torch.logsumexp(sg, 0) - torch.logsumexp(pos, 0)
        if "ranknet" in kinds:
            rank = rank + F.softplus(neg[None, :] - pos[:, None]).mean()
        if "lambda" in kinds:
            lam = lam + lambda_loss(sg, yg, a.lambda_k, a.lambda_misordered)
    if ng and kinds & {"pp", "list", "ranknet"}:
        parts["rank"] = rank / ng
        total = total + parts["rank"]
    if ng and "lambda" in kinds:
        parts["lambda"] = lam / ng
        total = total + a.lambda_alpha * parts["lambda"]
    return total, parts


def train_fold(a, model, enc, tr_ids, tr_y, tr_gid, device, amp_dtype, log_prefix, mid_eval=None, epoch_eval=None):
    groups = {}
    for i, g in enumerate(tr_gid):
        groups.setdefault(g, []).append(i)
    groups = list(groups.values())
    rng = random.Random(a.seed)
    params = [p for p in model.parameters() if p.requires_grad]
    n_tr = sum(p.numel() for p in params); n_all = sum(p.numel() for p in model.parameters())
    print(f"{log_prefix} параметров {n_all / 1e6:.0f}M, обучаемых (LoRA) {n_tr / 1e6:.1f}M; запросов {len(groups)}", flush=True)
    opt = torch.optim.AdamW(params, lr=a.lr, weight_decay=a.wd, betas=(0.9, 0.999), **({"fused": True} if device.type == "cuda" else {}))
    micro = [list(range(i, min(i + a.q_per_micro, len(groups)))) for i in range(0, len(groups), a.q_per_micro)]
    accum = max(1, a.q_per_step // a.q_per_micro)
    steps_ep = math.ceil(len(micro) / accum)
    total = steps_ep * a.epochs if not a.dry_run else max(1, math.ceil(a.dry_run / accum))
    from transformers import get_cosine_schedule_with_warmup, get_linear_schedule_with_warmup
    mk = get_cosine_schedule_with_warmup if a.sched == "cosine" else get_linear_schedule_with_warmup
    sched = mk(opt, max(1, int(a.warmup * total)), total)
    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats()
    t0, step, ntok = time.time(), 0, 0
    eval_at = set(int(len(micro) * (k + 1) / (a.eval_mid + 1)) for k in range(a.eval_mid)) if a.eval_mid else set()
    for ep in range(a.epochs):
        model.train()
        order = list(range(len(groups))); rng.shuffle(order)
        if a.dry_run:
            order.sort(key=lambda gi: -min(len(groups[gi]), a.max_docs or 10**9) * max(len(tr_ids[i]) for i in groups[gi]))
        run = {}; nb = 0
        opt.zero_grad(set_to_none=True)
        for bi in range(len(micro)):
            if a.dry_run and bi >= a.dry_run:
                break
            idx, gl = [], []
            for k, gi in enumerate(order[j] for j in micro[bi]):
                g = groups[gi]
                if a.max_docs and len(g) > a.max_docs:
                    pos = [i for i in g if tr_y[i] > 0][:a.max_docs - 1]
                    neg = [i for i in g if tr_y[i] <= 0]; rng.shuffle(neg)
                    g = pos + neg[:a.max_docs - len(pos)]
                idx += g; gl += [k] * len(g)
            batch = [tr_ids[i] for i in idx]
            ntok += len(batch) * max(len(x) for x in batch)
            s = forward_scores(model, enc, batch, device, amp_dtype)
            y = torch.tensor(tr_y[idx], dtype=torch.float32, device=device)
            loss, parts = group_loss(s, y, torch.tensor(gl, device=device), a)
            if not torch.is_tensor(loss):
                continue
            (loss / accum).backward()
            for k, v in parts.items():
                run[k] = run.get(k, 0.0) + v.detach().item()
            nb += 1
            if (bi + 1) % accum == 0 or bi + 1 == len(micro) or (a.dry_run and bi + 1 == a.dry_run):
                torch.nn.utils.clip_grad_norm_(params, a.clip)
                opt.step(); sched.step(); opt.zero_grad(set_to_none=True); step += 1
            if a.log_every and (bi + 1) % a.log_every == 0:
                el = time.time() - t0
                mem = f" mem {torch.cuda.max_memory_allocated() / 2**30:.1f}GB" if device.type == "cuda" else ""
                print(f"{log_prefix} ep{ep + 1} {bi + 1}/{len(micro)} step {step}/{total} "
                      + " ".join(f"{k} {v / nb:.4f}" for k, v in run.items())
                      + f" lr {sched.get_last_lr()[0]:.2e} {ntok / el:.0f} tok/s {el:.0f}s ETA {el / (bi + 1) * (len(micro) - bi - 1):.0f}s{mem}", flush=True)
            if mid_eval is not None and (bi + 1) in eval_at:
                print(f"{log_prefix} промежуточный val NDCG@10 (подвыборка) {mid_eval(model):.4f} на {bi + 1}/{len(micro)}", flush=True)
                model.train()
        msg = f"{log_prefix} ep{ep + 1} " + " ".join(f"{k} {v / max(1, nb):.4f}" for k, v in run.items()) + f" time {time.time() - t0:.0f}s"
        if device.type == "cuda":
            msg += f" peak_mem {torch.cuda.max_memory_allocated() / 2**30:.1f}GB"
        if epoch_eval is not None and ep + 1 < a.epochs and not a.dry_run:
            msg += f" val NDCG@10 {epoch_eval(model):.4f}"
        print(msg, flush=True)
    return model


def get_args(argv=None):
    ap = argparse.ArgumentParser()
    ap.add_argument("--profile", default=None, help="имя профиля из --config (значения по умолчанию; CLI главнее)")
    ap.add_argument("--config", default="configs/llm_rerank.json")
    ap.add_argument("--data", default="data")
    ap.add_argument("--out", default="artifacts")
    ap.add_argument("--model", default="Qwen/Qwen3-Reranker-4B")
    ap.add_argument("--prompt_style", default="auto", choices=["auto", "qwen3", "bge_gemma"])
    ap.add_argument("--instruction", default=DEFAULT_INSTRUCTION)
    ap.add_argument("--tag", default=None)
    ap.add_argument("--folds", type=int, default=5)
    ap.add_argument("--only_folds", type=int, nargs="*", default=None)
    ap.add_argument("--max_len", type=int, default=768)
    ap.add_argument("--epochs", type=int, default=1)
    ap.add_argument("--lr", type=float, default=1e-4)
    ap.add_argument("--wd", type=float, default=0.0)
    ap.add_argument("--warmup", type=float, default=0.05)
    ap.add_argument("--sched", default="cosine", choices=["cosine", "linear"])
    ap.add_argument("--clip", type=float, default=1.0)
    ap.add_argument("--lora_r", type=int, default=16)
    ap.add_argument("--lora_alpha", type=int, default=32)
    ap.add_argument("--lora_dropout", type=float, default=0.05)
    ap.add_argument("--lora_targets", default="q_proj,k_proj,v_proj,o_proj,gate_proj,up_proj,down_proj")
    ap.add_argument("--loss", default="pp+bce", help="комбинация через '+': pp, list, ranknet, lambda (= lambdarank), bce; "
                                                    "напр. lambdarank, lambda+bce, pp+lambda, pp+lambda+bce")
    ap.add_argument("--lambda_alpha", type=float, default=1.0, help="вес LambdaRank-слагаемого (α из sem11_2)")
    ap.add_argument("--lambda_k", type=int, default=10, help="отсечка NDCG@k в весах LambdaRank")
    ap.add_argument("--lambda_misordered", action="store_true", help="вес только неверно упорядоченным парам")
    ap.add_argument("--bce_w", type=float, default=0.5)
    ap.add_argument("--label_smooth", type=float, default=0.0)
    ap.add_argument("--tau", type=float, default=1.0, help="температура для pp/list/ranknet")
    ap.add_argument("--q_per_micro", type=int, default=1, help="запросов в одном forward")
    ap.add_argument("--q_per_step", type=int, default=8, help="запросов на шаг оптимизатора (grad accumulation)")
    ap.add_argument("--max_docs", type=int, default=16, help="макс. кандидатов запроса в обучении (все позитивы + случайные негативы)")
    ap.add_argument("--infer_tok_budget", type=int, default=65536, help="токенов в пакете инференса (n * длина)")
    ap.add_argument("--dtype", default="bf16", choices=["bf16", "fp16", "fp32"])
    ap.add_argument("--attn", default="sdpa", choices=["sdpa", "flash_attention_2", "eager"])
    ap.add_argument("--grad_ckpt", action=argparse.BooleanOptionalAction, default=True)
    ap.add_argument("--device", default="auto")
    ap.add_argument("--seed", type=int, default=42)
    ap.add_argument("--eval_before", action="store_true", help="перед обучением: zero-shot val NDCG фолда (+ fold{k}_val_zs.npy)")
    ap.add_argument("--eval_mid", type=int, default=0, help="K промежуточных оценок на подвыборке val (300 запросов)")
    ap.add_argument("--zero_shot", action="store_true", help="без обучения: скоры базовой модели на всём train/test -> zs_<tag>_{train,test}.npy")
    ap.add_argument("--save_adapter", action="store_true", help="сохранять LoRA-адаптер фолда (десятки МБ)")
    ap.add_argument("--max_queries", type=int, default=0, help="отладка: подвыборка запросов train для обучения")
    ap.add_argument("--dry_run", type=int, default=0, help="N микрошагов на фолде 0 + инференс 3 запросов + save/load адаптера, затем выход")
    ap.add_argument("--clean_v2", action="store_true", help="исправленная очистка заголовков (AMML_CLEAN_V2=1)")
    ap.add_argument("--log_every", type=int, default=50)
    pre, _ = ap.parse_known_args(argv)
    if pre.profile:
        prof = json.load(open(pre.config))[pre.profile]
        unknown = set(prof) - {x.dest for x in ap._actions}
        assert not unknown, f"неизвестные ключи профиля: {unknown}"
        ap.set_defaults(**prof)
    return ap.parse_args(argv)


def main(argv=None):
    a = get_args(argv)
    if a.clean_v2:
        os.environ["AMML_CLEAN_V2"] = "1"
    from transformers import AutoTokenizer
    if a.device == "auto":
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    else:
        device = torch.device(a.device)
    if device.type != "cuda" and a.dtype != "fp32":
        print(f"device={device}: dtype {a.dtype} -> fp32", flush=True); a.dtype = "fp32"
    amp_dtype = {"bf16": torch.bfloat16, "fp16": torch.float16}.get(a.dtype) if device.type == "cuda" else None
    style = style_of(a)
    tag = a.tag or f"{a.model.split('/')[-1].lower()}_lora_len{a.max_len}_s{a.seed}"
    out = os.path.join(a.out, f"ce_{tag}" + ("_dry" if a.dry_run else ""))
    if not a.zero_shot:
        os.makedirs(out, exist_ok=True)
    print(f"device={device} dtype={a.dtype} style={style} tag={tag} clean_v2={clean_v2()} torch={torch.__version__}", flush=True)

    tr, te = load_data(a.data)
    fp = os.path.join(a.out, "folds.npy")
    assert os.path.exists(fp), f"нет {fp}: скопируйте artifacts_colab/folds.npy (разбиение обязано совпадать со стеком)"
    folds = np.load(fp)
    assert len(folds) == len(tr) and int(folds.max()) + 1 == a.folds, "folds.npy не соответствует train.csv"
    tok = AutoTokenizer.from_pretrained(a.model)
    if tok.pad_token_id is None:
        tok.pad_token = tok.eos_token
    enc = Encoder(tok, style, a.max_len, a.instruction)
    t0 = time.time()
    tr_ids = enc(list(tr["query"]), [build_passage(t, x) for t, x in zip(tr["title"], tr["text"])])
    te_ids = enc(list(te["query"]), [build_passage(t, x) for t, x in zip(te["title"], te["text"])])
    lens = np.array([len(x) for x in tr_ids])
    print(f"токенизация {time.time() - t0:.0f}s; токенов на пару: mean {lens.mean():.0f}, p90 {np.percentile(lens, 90):.0f}, "
          f"max {lens.max()}, на пределе {a.max_len}: {(lens >= a.max_len).mean():.1%}; yes={enc.yes} no={enc.no}", flush=True)
    y = tr["relevance"].values.astype(np.float32)
    gid = tr["query_id"].values

    if a.max_queries:
        keep = tr["query_id"].isin(tr["query_id"].drop_duplicates().sample(a.max_queries, random_state=0)).values
    else:
        keep = np.ones(len(tr), dtype=bool)
    if a.zero_shot:
        model = load_model(a, device)
        rows = np.where((np.isin(folds, a.only_folds) if a.only_folds is not None else True) & keep)[0]
        ptr = np.full(len(tr), np.nan, dtype=np.float32)
        ptr[rows] = predict(model, enc, [tr_ids[i] for i in rows], device, amp_dtype, a.infer_tok_budget, log="[zero-shot train]")
        print(f"zero-shot NDCG@10 на {len(rows)} строках train: {score(tr.iloc[rows], ptr[rows]):.4f}; по фолдам "
              + str([round(score(tr[folds == f], ptr[folds == f]), 4) for f in range(a.folds) if np.isfinite(ptr[folds == f]).all()]), flush=True)
        np.save(os.path.join(a.out, f"zs_{tag}_train.npy"), ptr)
        if a.only_folds is None and not a.max_queries:
            np.save(os.path.join(a.out, f"zs_{tag}_test.npy"), predict(model, enc, te_ids, device, amp_dtype, a.infer_tok_budget, log="[zero-shot test]"))
        print("сохранено", os.path.join(a.out, f"zs_{tag}_*.npy")); return

    for f in range(a.folds):
        if a.only_folds is not None and f not in a.only_folds:
            continue
        vp, tp = os.path.join(out, f"fold{f}_val.npy"), os.path.join(out, f"fold{f}_test.npy")
        if os.path.exists(vp) and os.path.exists(tp) and not a.dry_run:
            print(f"[fold {f}] уже готов, пропуск"); continue
        set_seed(a.seed + f)
        tri = np.where((folds != f) & keep)[0]
        vai = np.where(folds == f)[0]
        model = add_lora(a, load_model(a, device))
        if a.dry_run:
            vq = tr.iloc[vai]["query_id"].drop_duplicates().values[:3]
            vai = vai[np.isin(gid[vai], vq)]
            te_rows = np.arange(min(20, len(te_ids)))
        else:
            te_rows = np.arange(len(te_ids))
        vsub = vai[np.isin(gid[vai], tr.iloc[vai]["query_id"].drop_duplicates().values[:300])]
        pred = lambda m, rows: predict(m, enc, [tr_ids[i] for i in rows], device, amp_dtype, a.infer_tok_budget)
        meta = {"tag": tag, "fold": f, "model": a.model, "style": style, "args": vars(a), "clean_v2": clean_v2()}
        if a.eval_before:
            zs = pred(model, vai); meta["ndcg10_zero_shot"] = score(tr.iloc[vai], zs)
            np.save(os.path.join(out, f"fold{f}_val_zs.npy"), zs)
            print(f"[fold {f}] zero-shot val NDCG@10 = {meta['ndcg10_zero_shot']:.4f}", flush=True)
        t0 = time.time()
        model = train_fold(a, model, enc, [tr_ids[i] for i in tri], y[tri], gid[tri], device, amp_dtype, f"[fold {f}]",
                           (lambda m: score(tr.iloc[vsub], pred(m, vsub))) if a.eval_mid else None,
                           lambda m: score(tr.iloc[vai], pred(m, vai)))
        meta["train_sec"] = time.time() - t0
        t0 = time.time()
        pv = pred(model, vai)
        meta["ndcg10"] = score(tr.iloc[vai], pv)
        print(f"[fold {f}] val NDCG@10 после обучения = {meta['ndcg10']:.4f}"
              + (f" (zero-shot {meta['ndcg10_zero_shot']:.4f}, Δ {meta['ndcg10'] - meta['ndcg10_zero_shot']:+.4f})" if a.eval_before else ""), flush=True)
        pt = predict(model, enc, [te_ids[i] for i in te_rows], device, amp_dtype, a.infer_tok_budget, log=f"[fold {f}] test")
        meta["infer_sec"] = time.time() - t0
        if device.type == "cuda":
            meta["peak_mem_gb"] = torch.cuda.max_memory_allocated() / 2**30
        if a.save_adapter or a.dry_run:
            ad = os.path.join(out, f"adapter_fold{f}")
            model.save_pretrained(ad)
            print(f"[fold {f}] адаптер: {ad} ({sum(os.path.getsize(os.path.join(ad, x)) for x in os.listdir(ad)) / 2**20:.1f} MB)", flush=True)
        if a.dry_run:
            from peft import PeftModel
            nb = sum(p.detach().float().abs().sum().item() for n, p in model.named_parameters() if "lora_B" in n)
            print(f"[dry] sum|lora_B| после обучения = {nb:.3e} (0 = градиенты не дошли)"
                  + (f"; max|скор после - zero-shot| = {np.abs(pv - zs).max():.3e}" if a.eval_before else ""), flush=True)
            assert nb > 0, "LoRA не обучается"
            del model
            m2 = PeftModel.from_pretrained(load_model(a, device), ad)
            pv2 = pred(m2, vai)
            print(f"[dry] save/load адаптера: max|Δ| = {np.abs(pv2 - pv).max():.2e}; val NDCG (3 запроса) {meta['ndcg10']:.4f}; "
                  f"test {pt.shape} finite={np.isfinite(pt).all()}", flush=True)
            assert np.abs(pv2 - pv).max() < 1e-2 and np.isfinite(pt).all()
            np.save(vp, pv); np.save(tp, pt)
            print("dry run ok"); return
        np.save(vp, pv); np.save(tp, pt)
        json.dump(meta, open(os.path.join(out, f"fold{f}_meta.json"), "w"), ensure_ascii=False, indent=1)
        del model
        if device.type == "cuda":
            torch.cuda.empty_cache()

    done = [f for f in range(a.folds) if os.path.exists(os.path.join(out, f"fold{f}_test.npy"))]
    if len(done) == a.folds:
        oof = np.zeros(len(tr), dtype=np.float32); fs = []
        for f in range(a.folds):
            vai = np.where(folds == f)[0]
            oof[vai] = np.load(os.path.join(out, f"fold{f}_val.npy")); fs.append(score(tr.iloc[vai], oof[vai]))
        test = np.mean([np.load(os.path.join(out, f"fold{f}_test.npy")) for f in range(a.folds)], axis=0)
        np.save(os.path.join(out, "oof.npy"), oof); np.save(os.path.join(out, "test.npy"), test)
        meta = {"tag": tag, "cv_mean": float(np.mean(fs)), "cv_std": float(np.std(fs)), "cv_folds": fs, "cv_all": score(tr, oof), "args": vars(a)}
        json.dump(meta, open(os.path.join(out, "meta.json"), "w"), ensure_ascii=False, indent=1)
        print(f"CV NDCG@10 [{tag}]: {meta['cv_mean']:.4f} ± {meta['cv_std']:.4f} (по фолдам {np.round(fs, 4).tolist()})")
    else:
        print(f"готовы фолды {done} из {a.folds}")


if __name__ == "__main__":
    main(sys.argv[1:])


Writing src/train_llm_rerank.py


In [13]:
%%writefile src/train_lora_t4.py


import argparse
import json
import math
import os
import random
import time

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

from .compat import hide_incompatible_torchao

hide_incompatible_torchao()

import numpy as np
import torch
import torch.distributed as dist

from .common import build_passage, load_data
from .metric import score
from .train_llm_rerank import DEFAULT_INSTRUCTION, Encoder, forward_scores, group_loss


def get_args(argv=None):
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", default="data")
    ap.add_argument("--out", default="artifacts")
    ap.add_argument("--model", default="Qwen/Qwen3-Reranker-4B")
    ap.add_argument("--tag", default="qwen4b_lora_t4")
    ap.add_argument("--instruction", default=DEFAULT_INSTRUCTION)
    ap.add_argument("--val_fold", type=int, default=0)
    ap.add_argument("--train_len", type=int, default=512)
    ap.add_argument("--infer_len", type=int, default=768)
    ap.add_argument("--max_docs", type=int, default=10)
    ap.add_argument("--q_per_step", type=int, default=8, help="запросов на шаг оптимизатора (суммарно по всем GPU)")
    ap.add_argument("--train_frac", type=float, default=1.0, help="доля train-запросов (для укладывания во время)")
    ap.add_argument("--lr", type=float, default=1e-4)
    ap.add_argument("--warmup", type=float, default=0.05)
    ap.add_argument("--lora_r", type=int, default=16)
    ap.add_argument("--lora_alpha", type=int, default=32)
    ap.add_argument("--lora_dropout", type=float, default=0.05)
    ap.add_argument("--lora_targets", default="q_proj,k_proj,v_proj,o_proj,gate_proj,up_proj,down_proj")
    ap.add_argument("--loss", default="pp+bce")
    ap.add_argument("--bce_w", type=float, default=0.5)
    ap.add_argument("--tok_budget", type=int, default=16384, help="токенов в пакете инференса")
    ap.add_argument("--ckpt_every", type=int, default=50, help="шагов оптимизатора между чекпоинтами")
    ap.add_argument("--seed", type=int, default=42)
    ap.add_argument("--log_every", type=int, default=10)
    ap.add_argument("--dry_steps", type=int, default=0, help=">0: N шагов обучения + инференс 3 val-запросов и 20 test-строк")
    ap.add_argument("--stop_after_steps", type=int, default=0, help="отладка: выйти после N шагов (имитация обрыва)")
    return ap.parse_args(argv)


def log(msg, rank, path=None):
    if rank == 0:
        print(msg, flush=True)
        if path:
            open(path, "a").write(msg + "\n")


@torch.no_grad()
def infer(model, enc, ids, device, amp, tok_budget, rank, world):

    model.eval()
    lens = np.array([len(x) for x in ids]); order = np.argsort(-lens, kind="stable")[rank::world]
    out = {}
    i = 0
    while i < len(order):
        n = max(1, min(256, tok_budget // int(lens[order[i]])))
        idx = order[i:i + n]
        s = forward_scores(model, enc, [ids[j] for j in idx], device, torch.float16 if amp else None).float().cpu().numpy()
        out.update(zip(idx.tolist(), s.tolist())); i += n
    if world > 1:
        parts = [None] * world; dist.all_gather_object(parts, out)
        for p in parts:
            out.update(p)
    res = np.full(len(ids), np.nan, dtype=np.float32)
    res[list(out)] = list(out.values())
    return res


def main(argv=None):
    a = get_args(argv)
    world, rank, local = int(os.environ.get("WORLD_SIZE", 1)), int(os.environ.get("RANK", 0)), int(os.environ.get("LOCAL_RANK", 0))
    cuda = torch.cuda.is_available()
    if world > 1:
        dist.init_process_group("nccl" if cuda else "gloo")
    device = torch.device(f"cuda:{local}" if cuda else "cpu")
    if cuda:
        torch.cuda.set_device(device)
    amp = cuda
    out = os.path.join(a.out, f"ce_{a.tag}"); os.makedirs(out, exist_ok=True)
    L = os.path.join(out, "train_log.txt")
    random.seed(a.seed); np.random.seed(a.seed); torch.manual_seed(a.seed + rank)

    tr, te = load_data(a.data)
    folds = np.load(os.path.join(a.out, "folds.npy")); assert len(folds) == len(tr)
    tri, vai = np.where(folds != a.val_fold)[0], np.where(folds == a.val_fold)[0]
    te_rows = np.arange(len(te))
    if a.dry_steps:
        vai = vai[np.isin(tr.query_id.values[vai], list(dict.fromkeys(tr.query_id.values[vai]))[:3])]
        te_rows = te_rows[:20]
    from transformers import AutoModelForCausalLM, AutoTokenizer, get_cosine_schedule_with_warmup
    tok = AutoTokenizer.from_pretrained(a.model)
    enc_tr, enc_inf = Encoder(tok, "qwen3", a.train_len, a.instruction), Encoder(tok, "qwen3", a.infer_len, a.instruction)
    P = lambda d, r: [build_passage(t, x) for t, x in zip(d["title"].values[r], d["text"].values[r])]
    t0 = time.time()
    tr_ids = enc_tr(list(tr["query"].values[tri]), P(tr, tri))
    va_ids = enc_inf(list(tr["query"].values[vai]), P(tr, vai))
    te_ids = enc_inf(list(te["query"].values[te_rows]), P(te, te_rows))
    y, gid = tr["relevance"].values[tri].astype(np.float32), tr["query_id"].values[tri]
    lens = np.array([len(x) for x in tr_ids])
    log(f"[{a.tag}] world={world} device={device} amp_fp16={amp}; токенизация {time.time() - t0:.0f}s; train пар {len(tri)} "
        f"(len {a.train_len}: mean {lens.mean():.0f}, на пределе {(lens >= a.train_len).mean():.1%}), val {len(vai)}, test {len(te_ids)}", rank, L)

    dt = torch.float16 if cuda else torch.float32
    try:
        model = AutoModelForCausalLM.from_pretrained(a.model, dtype=dt, attn_implementation="sdpa")
    except TypeError:
        model = AutoModelForCausalLM.from_pretrained(a.model, torch_dtype=dt, attn_implementation="sdpa")
    model.config.use_cache = False
    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    from peft import LoraConfig, get_peft_model, get_peft_model_state_dict, set_peft_model_state_dict
    model = get_peft_model(model, LoraConfig(r=a.lora_r, lora_alpha=a.lora_alpha, lora_dropout=a.lora_dropout, bias="none", task_type="CAUSAL_LM",
                                             target_modules=a.lora_targets.split(",")))
    for p in model.parameters():
        if p.requires_grad:
            p.data = p.data.float()
    model.to(device)
    tp = [p for p in model.parameters() if p.requires_grad]
    log(f"  обучаемых {sum(p.numel() for p in tp) / 1e6:.1f}M (dtype {tp[0].dtype}), база {dt}", rank, L)
    core = model
    if world > 1:
        model = torch.nn.parallel.DistributedDataParallel(model, device_ids=[local] if cuda else None)

    groups = {}
    for i, g in enumerate(gid):
        groups.setdefault(g, []).append(i)
    groups = list(groups.values())
    rng = random.Random(a.seed); rng.shuffle(groups)
    if a.train_frac < 1:
        groups = groups[:int(len(groups) * a.train_frac)]
    accum = max(1, a.q_per_step // world)
    n_steps = len(groups) // (accum * world)
    if a.dry_steps:
        n_steps = min(n_steps, a.dry_steps)
    groups = groups[:n_steps * accum * world]
    my = groups[rank::world]
    opt = torch.optim.AdamW(tp, lr=a.lr, weight_decay=0.0)
    sched = get_cosine_schedule_with_warmup(opt, max(1, int(a.warmup * n_steps)), n_steps)
    scaler = torch.amp.GradScaler("cuda", enabled=amp)
    step, skipped = 0, 0
    ck = os.path.join(out, "ckpt_last.pt")
    if os.path.exists(ck):
        st = torch.load(ck, map_location="cpu", weights_only=False)
        set_peft_model_state_dict(core, st["adapter"]); opt.load_state_dict(st["opt"]); sched.load_state_dict(st["sched"])
        scaler.load_state_dict(st["scaler"]); step, skipped = st["step"], st["skipped"]
        for p in tp:
            p.data = p.data.float()
        log(f"  продолжение с чекпоинта: шаг {step}/{n_steps}", rank, L)
    a_loss = argparse.Namespace(loss=a.loss, bce_w=a.bce_w, label_smooth=0.0, tau=1.0, lambda_alpha=1.0, lambda_k=10, lambda_misordered=False)
    model.train(); t0 = time.time(); run = {}; nb = 0; tok_seen = 0; step0 = step
    while step < n_steps:
        opt.zero_grad(set_to_none=True)
        for k in range(accum):
            g = my[step * accum + k]
            if len(g) > a.max_docs:
                pos = [i for i in g if y[i] > 0][:a.max_docs - 1]; neg = [i for i in g if y[i] <= 0]
                random.Random(a.seed * 100003 + rank * 7919 + step * accum + k).shuffle(neg); g = pos + neg[:a.max_docs - len(pos)]
            batch = [tr_ids[i] for i in g]; tok_seen += len(batch) * max(len(x) for x in batch)
            ctx = model.no_sync() if (world > 1 and k < accum - 1) else torch.enable_grad()
            with ctx:
                s = forward_scores(model, enc_tr, batch, device, torch.float16 if amp else None).float()
                loss, parts = group_loss(s, torch.tensor(y[g], device=device), torch.zeros(len(g), dtype=torch.long, device=device), a_loss)
                scaler.scale(loss / accum).backward()
            for kk, v in parts.items():
                run[kk] = run.get(kk, 0.0) + float(v.detach())
            nb += 1
        scaler.unscale_(opt)
        gn = torch.nn.utils.clip_grad_norm_(tp, 1.0)
        sc0 = scaler.get_scale() if amp else 1.0
        scaler.step(opt); scaler.update()
        if amp and scaler.get_scale() < sc0:
            skipped += 1
        sched.step(); step += 1
        if step % a.log_every == 0 or step == n_steps:
            el = time.time() - t0
            log(f"  шаг {step}/{n_steps} " + " ".join(f"{kk} {v / max(1, nb):.4f}" for kk, v in run.items())
                + f" |g| {float(gn):.2f} scale {sc0:.0f} пропущено {skipped} {tok_seen * world / el:.0f} ток/с, {el:.0f}s, "
                f"ETA {el / max(1, step - step0) * (n_steps - step) / 60:.0f} мин"
                + (f", mem {torch.cuda.max_memory_allocated(device) / 2**30:.1f}GB" if cuda else ""), rank, L)
            run, nb = {}, 0
        if rank == 0 and (step % a.ckpt_every == 0 or step == n_steps):
            torch.save({"adapter": get_peft_model_state_dict(core), "opt": opt.state_dict(), "sched": sched.state_dict(),
                        "scaler": scaler.state_dict(), "step": step, "skipped": skipped}, ck + ".tmp")
            os.replace(ck + ".tmp", ck)
        if a.stop_after_steps and step >= a.stop_after_steps and step < n_steps:
            log(f"  (остановка по --stop_after_steps на шаге {step})", rank, L)
            if world > 1:
                dist.barrier(); dist.destroy_process_group()
            return
    if skipped > 0.2 * max(1, n_steps):
        log(f"  !!! GradScaler пропустил {skipped} из {n_steps} шагов — нестабильность fp16", rank, L)
    t1 = time.time()
    pv = infer(core, enc_inf, va_ids, device, amp, a.tok_budget, rank, world)
    nd = score(tr.iloc[vai], pv)
    log(f"[{a.tag}] val NDCG@10 (фолд {a.val_fold}, после эпохи) = {nd:.4f}; NaN/inf {int((~np.isfinite(pv)).sum())}; инференс val {time.time() - t1:.0f}s", rank, L)
    t2 = time.time()
    pt = infer(core, enc_inf, te_ids, device, amp, a.tok_budget, rank, world)
    if rank == 0:
        np.save(os.path.join(out, f"fold{a.val_fold}_val.npy"), pv); np.save(os.path.join(out, "test.npy"), pt)
        core.save_pretrained(os.path.join(out, "adapter"))
        json.dump({"tag": a.tag, "model": a.model, "val_fold": a.val_fold, "ndcg10_val": nd, "n_steps": n_steps, "skipped_steps": skipped,
                   "train_sec": t1 - t0, "infer_val_sec": t2 - t1, "infer_test_sec": time.time() - t2, "val_nonfinite": int((~np.isfinite(pv)).sum()),
                   "test_nonfinite": int((~np.isfinite(pt)).sum()), "args": vars(a), "world": world, "dry": bool(a.dry_steps)},
                  open(os.path.join(out, "meta.json"), "w"), ensure_ascii=False, indent=1)
        log(f"[{a.tag}] сохранено: fold{a.val_fold}_val.npy {pv.shape}, test.npy {pt.shape}, adapter/, meta.json; test {time.time() - t2:.0f}s", rank, L)
    if world > 1:
        dist.barrier(); dist.destroy_process_group()


if __name__ == "__main__":
    main()


Writing src/train_lora_t4.py


In [14]:
%%writefile src/stack.py

import argparse
import glob
import json
import os

import numpy as np
import pandas as pd

from .features import group_feats
from .metric import fold_scores, score

META = ["id", "query_id", "document_id", "article_id", "relevance"]


def zq(df, v):
    g = pd.Series(v).groupby(df["query_id"].values)
    return ((pd.Series(v) - g.transform("mean")) / (g.transform("std").fillna(0) + 1e-6)).values


def load_ce(out, tag, folds, allow_partial=False):


    d = os.path.join(out, f"ce_{tag}")
    nf = int(folds.max()) + 1
    if os.path.exists(os.path.join(d, "oof.npy")) and os.path.exists(os.path.join(d, "test.npy")):
        return np.load(os.path.join(d, "oof.npy")).astype(np.float64), np.load(os.path.join(d, "test.npy")).astype(np.float64), list(range(nf))
    done = [f for f in range(nf) if os.path.exists(os.path.join(d, f"fold{f}_val.npy")) and os.path.exists(os.path.join(d, f"fold{f}_test.npy"))]
    if not allow_partial or not done:
        raise FileNotFoundError(f"{d}: нет полного OOF (готовы фолды {done}); для частичной модели добавьте --allow_partial")
    oof = np.full(len(folds), np.nan)
    for f in done:
        v = np.load(os.path.join(d, f"fold{f}_val.npy"))
        assert len(v) == (folds == f).sum(), f"{tag}: размер fold{f}_val.npy не совпадает с folds.npy"
        oof[folds == f] = v
    test = np.mean([np.load(os.path.join(d, f"fold{f}_test.npy")) for f in done], axis=0).astype(np.float64)
    return oof, test, done


def extra_paths(path):

    p = path[:-len("_train.npy")] if path.endswith("_train.npy") else path
    return p + "_train.npy", p + "_test.npy"


def add_extra(a, ftr, fte):


    names, ok = [], np.ones(len(ftr), dtype=bool)
    for path, name in a.extra_feat:
        ptr, pte = extra_paths(path)
        vtr, vte = np.load(ptr).astype(np.float64).ravel(), np.load(pte).astype(np.float64).ravel()
        assert name not in ftr.columns, f"--extra_feat: имя {name} уже занято"
        assert len(vtr) == len(ftr) and len(vte) == len(fte), f"{name}: длины {len(vtr)}/{len(vte)} != {len(ftr)}/{len(fte)}"
        assert np.isfinite(vte).all(), f"{name}: в test есть NaN — скор должен быть посчитан на всём test"
        fin = np.isfinite(vtr)
        whole = pd.Series(fin).groupby(ftr["query_id"].values).transform(lambda s: s.all() or not s.any()).all()
        assert whole, f"{name}: NaN в train должны покрывать запросы целиком"
        if not fin.all() and not a.extra_rows_only:
            raise ValueError(f"{name}: скор есть только у {fin.mean():.1%} строк train; для отладки добавьте --extra_rows_only")
        ftr[name], fte[name] = vtr, vte
        ok &= fin
        names.append(name)
        print(f"внешний признак {name}: {ptr} (+ _test), строк train со скором {fin.sum()} из {len(fin)}")
    return names, ok


def load(a):
    ftr = pd.read_pickle(os.path.join(a.out, "feats_train.pkl"))
    fte = pd.read_pickle(os.path.join(a.out, "feats_test.pkl"))
    folds = np.load(os.path.join(a.out, "folds.npy"))
    extra_cols, extra_ok = add_extra(a, ftr, fte)
    tags = a.ce
    if tags == ["all"]:
        pat = "fold*_val.npy" if a.allow_partial else "oof.npy"
        tags = sorted({os.path.basename(os.path.dirname(p))[3:] for p in glob.glob(os.path.join(a.out, "ce_*", pat))})
    elif tags == ["none"]:
        tags = []
    ce_cols = []
    use_folds = set(range(int(folds.max()) + 1))
    for t in tags:
        c = "ce_" + t.replace("-", "_").replace(".", "_")
        ftr[c], fte[c], done = load_ce(a.out, t, folds, a.allow_partial)
        use_folds &= set(done)
        ce_cols.append(c)
    if a.only_folds:
        use_folds &= set(a.only_folds)
    use_folds = sorted(use_folds)
    assert len(use_folds) >= 2, f"общих готовых фолдов меньше двух: {use_folds}"
    if len(use_folds) <= int(folds.max()) or not extra_ok.all():


        keep = np.isin(folds, use_folds) & extra_ok
        print(f"ЧАСТИЧНЫЙ РЕЖИМ: фолды {use_folds}, строк train {keep.sum()} из {len(keep)} ({keep.mean():.0%})")
        ftr["_row"] = np.arange(len(ftr))
        ftr, folds = ftr[keep].reset_index(drop=True), folds[keep]
        assert not ftr[ce_cols + extra_cols].isna().any().any()
    else:
        ftr["_row"] = np.arange(len(ftr))
    for c in extra_cols:
        ftr, fte = group_feats(ftr, c), group_feats(fte, c)
    if a.extra_mix and extra_cols and ce_cols:
        for df in (ftr, fte):
            df["mix_z"] = np.mean([zq(df, df[c].values) for c in ce_cols + extra_cols], axis=0)
        ftr, fte = group_feats(ftr, "mix_z"), group_feats(fte, "mix_z")
    if len(ce_cols) > 1:
        for df in (ftr, fte):
            df["ce_mean_z"] = np.mean([zq(df, df[c].values) for c in ce_cols], axis=0)
        ce_all = ce_cols + ["ce_mean_z"]
    else:
        ce_all = list(ce_cols)
    for c in ce_all:
        ftr, fte = group_feats(ftr, c), group_feats(fte, c)
    if ce_all:
        c = ce_all[-1]
        for df in (ftr, fte):
            g = df.groupby(["query_id", "article_id"])[c]
            df["ce_art_max"] = g.transform("max"); df["ce_art_mean"] = g.transform("mean")
    feat_cols = [c for c in ftr.columns if c not in META + ["_row"]]
    if a.drop_emb:
        feat_cols = [c for c in feat_cols if not c.startswith("emb_")]
    return ftr, fte, folds, feat_cols, ce_cols, use_folds


def fit_predict(Xtr, ytr, gtr, Xva, yva, gva, params, rounds, seed):
    import lightgbm as lgb
    p = dict(params, seed=seed, bagging_seed=seed, feature_fraction_seed=seed)
    dtr = lgb.Dataset(Xtr, ytr, group=gtr)
    valid = [lgb.Dataset(Xva, yva, group=gva, reference=dtr)] if Xva is not None else []
    ev = {}
    m = lgb.train(p, dtr, rounds, valid_sets=valid, valid_names=["val"], callbacks=[lgb.record_evaluation(ev)] if valid else [])
    return m, (np.array(ev["val"]["ndcg@10"]) if valid else None)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--out", default="artifacts")
    ap.add_argument("--ce", nargs="*", default=["all"])
    ap.add_argument("--tag", default="main")
    ap.add_argument("--seeds", type=int, default=3)
    ap.add_argument("--rounds", type=int, default=600)
    ap.add_argument("--lr", type=float, default=0.03)
    ap.add_argument("--leaves", type=int, default=15)
    ap.add_argument("--drop_emb", action="store_true")
    ap.add_argument("--only_folds", type=int, nargs="*", default=None,
                    help="ограничить стек строками этих фолдов (напр. для сравнения без CE на тех же строках)")
    ap.add_argument("--allow_partial", action="store_true",
                    help="разрешить модели с неполным набором фолдов: стек только на строках общих готовых фолдов")
    ap.add_argument("--extra_feat", nargs=2, action="append", default=[], metavar=("PATH", "NAME"),
                    help="внешний скор без фолдов (zero-shot): PATH = <prefix>_train.npy или <prefix> (test = <prefix>_test.npy); "
                         "добавляются NAME, NAME_rank, NAME_z, NAME_dmax. Можно повторять")
    ap.add_argument("--extra_mix", action="store_true", help="+ признак mix_z = среднее z(CE) и z(внешних скоров) и его rank/z/dmax")
    ap.add_argument("--extra_rows_only", action="store_true",
                    help="отладка: внешний скор есть не у всех запросов train -> стек только на этих запросах")
    a = ap.parse_args()

    ftr, fte, folds, cols, ce_cols, fold_ids = load(a)
    n_train_total = len(np.load(os.path.join(a.out, "folds.npy")))

    otr = np.argsort(ftr["query_id"].values, kind="mergesort"); ote = np.argsort(fte["query_id"].values, kind="mergesort")
    ftr_s, fte_s, folds_s = ftr.iloc[otr].reset_index(drop=True), fte.iloc[ote].reset_index(drop=True), folds[otr]
    y = ftr_s["relevance"].values
    params = dict(objective="lambdarank", metric="ndcg", ndcg_eval_at=[10], learning_rate=a.lr, num_leaves=a.leaves,
                  min_data_in_leaf=40, feature_fraction=0.6, bagging_fraction=0.8, bagging_freq=1, lambda_l2=5.0,
                  lambdarank_truncation_level=15, verbose=-1, num_threads=0)
    gsz = lambda d: d.groupby("query_id", sort=False).size().values
    print(f"признаков: {len(cols)}; CE: {ce_cols}")
    for c in ce_cols:
        fs = fold_scores(ftr, ftr[c].values, folds)
        print(f"  одиночный {c}: {np.mean(fs):.4f} ± {np.std(fs):.4f}")
    if len(ce_cols) > 1:
        fs = fold_scores(ftr, ftr["ce_mean_z"].values, folds)
        print(f"  среднее z-скоров CE: {np.mean(fs):.4f} ± {np.std(fs):.4f}")
    for c in [n for _, n in a.extra_feat] + (["mix_z"] if "mix_z" in ftr.columns else []):
        fs = fold_scores(ftr, ftr[c].values, folds)
        print(f"  одиночный внешний {c}: {np.mean(fs):.4f} ± {np.std(fs):.4f}")

    nf = len(fold_ids)
    curves = np.zeros((nf, a.rounds))
    for ci, f in enumerate(fold_ids):
        tr_m, va_m = folds_s != f, folds_s == f
        _, ev = fit_predict(ftr_s.loc[tr_m, cols], y[tr_m], gsz(ftr_s[tr_m]), ftr_s.loc[va_m, cols], y[va_m], gsz(ftr_s[va_m]), params, a.rounds, 0)
        curves[ci] = ev
    mean_curve = curves.mean(0)
    sm = np.convolve(mean_curve, np.ones(21) / 21, mode="same")
    best = int(np.argmax(sm[10:-10]) + 10) + 1
    print(f"лучшее число раундов (сглаж. средняя кривая): {best}; lgb-ndcg@10 там {mean_curve[best - 1]:.4f}")
    oof = np.zeros(len(ftr_s)); test = np.zeros(len(fte_s)); imp = np.zeros(len(cols))
    for f in fold_ids:
        tr_m, va_m = folds_s != f, folds_s == f
        for s in range(a.seeds):
            m, _ = fit_predict(ftr_s.loc[tr_m, cols], y[tr_m], gsz(ftr_s[tr_m]), None, None, None, params, best, s)
            oof[va_m] += m.predict(ftr_s.loc[va_m, cols]) / a.seeds
            test += m.predict(fte_s[cols]) / (a.seeds * nf)
            imp += m.feature_importance("gain")
    fs = fold_scores(ftr_s, oof, folds_s)
    print(f"СТЕК [{a.tag}] CV NDCG@10: {np.mean(fs):.4f} ± {np.std(fs):.4f} (фолды {np.round(fs, 4).tolist()}), по всем запросам {score(ftr_s, oof):.4f}")
    print("топ-15 признаков по gain:", ", ".join(pd.Series(imp, cols).sort_values(ascending=False).head(15).index))
    oof_o = np.full(n_train_total, np.nan); oof_o[ftr_s["_row"].values] = oof
    test_o = np.zeros_like(test); test_o[ote] = test
    np.save(os.path.join(a.out, f"stack_{a.tag}_oof.npy"), oof_o); np.save(os.path.join(a.out, f"stack_{a.tag}_test.npy"), test_o)
    json.dump({"tag": a.tag, "cv_mean": float(np.mean(fs)), "cv_std": float(np.std(fs)), "cv_folds": fs, "rounds": best,
               "ce": ce_cols, "extra": [list(x) for x in a.extra_feat], "extra_mix": a.extra_mix, "n_features": len(cols), "seeds": a.seeds, "folds_used": [int(f) for f in fold_ids],
               "n_train_rows": int(len(ftr_s)), "partial": bool(len(ftr_s) < n_train_total)},
              open(os.path.join(a.out, f"stack_{a.tag}_meta.json"), "w"), indent=1)


if __name__ == "__main__":
    main()


Writing src/stack.py


In [15]:
%%writefile src/predict.py

import argparse
import os

import numpy as np
import pandas as pd


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", default="data")
    ap.add_argument("--pred", nargs="+", required=True)
    ap.add_argument("--weights", nargs="*", type=float, default=None)
    ap.add_argument("--out_csv", default="submission.csv")
    a = ap.parse_args()
    te = pd.read_csv(os.path.join(a.data, "test.csv"), usecols=["id", "query_id"])
    ss = pd.read_csv(os.path.join(a.data, "sample_submission.csv"))
    preds = [np.load(p).astype(np.float64) for p in a.pred]
    assert all(len(p) == len(te) for p in preds)
    if len(preds) == 1:
        final = preds[0]
    else:
        w = a.weights or [1.0] * len(preds)
        final = np.zeros(len(te))
        for p, wi in zip(preds, w):
            s = pd.Series(p); g = s.groupby(te["query_id"].values)
            final += wi * ((s - g.transform("mean")) / (g.transform("std").fillna(0) + 1e-9)).values
    assert np.isfinite(final).all()
    te["prediction"] = final
    ties = int(te.duplicated(["query_id", "prediction"]).sum())
    if ties:
        rng = np.random.RandomState(0)
        te["prediction"] += rng.uniform(0, 1e-6, len(te))
    sub = ss[["id"]].merge(te[["id", "prediction"]], on="id", how="left")
    assert sub["prediction"].notna().all() and len(sub) == len(ss)
    sub.to_csv(a.out_csv, index=False)
    print(f"{a.out_csv}: {len(sub)} строк, ничьих внутри запроса было {ties}, уникальных значений {sub.prediction.nunique()}")


if __name__ == "__main__":
    main()


Writing src/predict.py


In [16]:
%%writefile jina/jina_zs.py

import argparse
import hashlib
import html
import json
import os
import re
import time
import unicodedata

import numpy as np
import pandas as pd

MODEL = "jinaai/jina-reranker-v2-base-multilingual"
_TAG = re.compile(r"</?[a-zA-Z][^<>]{0,200}>")
_CTRL = re.compile("[\x00-\x08\x0b\x0c\x0e-\x1f\x7f­​-‏⁠﻿̀́]")
_WS = re.compile(r"\s+")
_MOJI = re.compile("[РС][\u0080-ӿ]")


def fix_mojibake(s):

    if len(s) < 4 or len(_MOJI.findall(s)) < 0.25 * len(s):
        return s
    try:
        return s.encode("cp1251").decode("utf-8")
    except Exception:
        return s


def clean_text(s):
    if not isinstance(s, str):
        return ""
    s = fix_mojibake(s)
    for _ in range(3):
        u = html.unescape(s)
        if u == s:
            break
        s = u
    s = _TAG.sub(" ", s)
    s = _CTRL.sub("", s)
    s = unicodedata.normalize("NFKC", s)
    return _WS.sub(" ", s).strip()


def clean_title(s):
    s = clean_text(s)
    if len(s) > 6 and s.isupper() and not any(ch.isdigit() for ch in s):
        s = s.capitalize()
    return s


def load(data_dir):
    tr = pd.read_csv(os.path.join(data_dir, "train.csv"))
    te = pd.read_csv(os.path.join(data_dir, "test.csv"))
    for df in (tr, te):
        df["query"] = df["query"].map(clean_text)
        df["title"] = df["title"].map(clean_title)
        df["text"] = df["text"].map(clean_text)
    return tr, te


def passage(title, text):
    return f"{title}. {text}" if title else text


def ndcg10_per_query(query_id, ids, rel, pred):

    qc, _ = pd.factorize(np.asarray(query_id))
    ids = np.asarray(ids).astype(str)
    rel = np.asarray(rel, dtype=float)
    id_rank = np.argsort(np.argsort(ids, kind="mergesort"), kind="mergesort")
    order = np.lexsort((id_rank, -np.asarray(pred, dtype=float), qc))
    qs = qc[order]
    start = np.r_[0, np.flatnonzero(np.diff(qs)) + 1]
    pos = np.arange(len(qs)) - np.repeat(start, np.diff(np.r_[start, len(qs)]))
    nq = qc.max() + 1
    dcg = np.bincount(qs, weights=rel[order] * (pos < 10) / np.log2(pos + 2), minlength=nq)
    cs = np.concatenate([[0.0], np.cumsum(1.0 / np.log2(np.arange(1, 11) + 1))])
    idcg = cs[np.minimum(np.bincount(qc, weights=rel, minlength=nq).astype(int), 10)]
    return np.where(idcg > 0, dcg / np.where(idcg > 0, idcg, 1), 0.0)


def make_batches(lens, order, tok_budget, max_bs):
    out, cur = [], []
    for j in order:
        if cur and ((len(cur) + 1) * lens[cur[0]] > tok_budget or len(cur) >= max_bs):
            out.append(cur)
            cur = []
        cur.append(j)
    if cur:
        out.append(cur)
    return out


def _shim_transformers():


    import torch
    import transformers.models.xlm_roberta.modeling_xlm_roberta as xr
    if hasattr(xr, "create_position_ids_from_input_ids"):
        return False

    def create_position_ids_from_input_ids(input_ids, padding_idx, past_key_values_length=0):
        mask = input_ids.ne(padding_idx).int()
        incremental_indices = (torch.cumsum(mask, dim=1).type_as(mask) + past_key_values_length) * mask
        return incremental_indices.long() + padding_idx

    xr.create_position_ids_from_input_ids = create_position_ids_from_input_ids
    return True


def get_model(fp16, device):
    import torch
    _shim_transformers()
    from transformers import AutoModelForSequenceClassification
    kw = dict(trust_remote_code=True, dtype=torch.float32)
    try:
        m = AutoModelForSequenceClassification.from_pretrained(MODEL, use_flash_attn=False, **kw)
    except TypeError:
        kw["torch_dtype"] = kw.pop("dtype")
        m = AutoModelForSequenceClassification.from_pretrained(MODEL, use_flash_attn=False, **kw)
    m = (m.half() if fp16 else m).to(device).eval()
    return m


def score_pairs(model, tok, queries, passages, max_len, tok_budget, max_bs, device, log_every=50, tag=""):

    import torch
    t0 = time.time()
    enc = tok(list(queries), list(passages), truncation="only_second", max_length=max_len, padding=False)["input_ids"]
    lens = np.array([len(x) for x in enc])
    print(f"{tag}токенизация {time.time() - t0:.0f}s; пар {len(enc)}; токенов mean {lens.mean():.0f}, p99 {np.percentile(lens, 99):.0f}, "
          f"max {lens.max()}, на пределе {max_len}: {(lens >= max_len).mean():.2%}", flush=True)
    order = np.argsort(-lens, kind="stable")
    out = np.full(len(enc), np.nan, dtype=np.float32)
    pad = tok.pad_token_id

    def fwd(b):
        L = int(max(lens[j] for j in b))
        ids = torch.full((len(b), L), pad, dtype=torch.long)
        att = torch.zeros((len(b), L), dtype=torch.long)
        for k, j in enumerate(b):
            ids[k, :lens[j]] = torch.tensor(enc[j])
            att[k, :lens[j]] = 1
        with torch.inference_mode():
            lg = model(input_ids=ids.to(device), attention_mask=att.to(device), return_dict=True).logits.view(-1).float().cpu().numpy()
        out[b] = lg

    def safe(b):
        try:
            fwd(b)
        except torch.OutOfMemoryError:
            if len(b) == 1:
                raise
            torch.cuda.empty_cache()
            print(f"  OOM на пакете {len(b)}x{lens[b[0]]} -> делю пополам", flush=True)
            safe(b[:len(b) // 2])
            safe(b[len(b) // 2:])

    batches = make_batches(lens, order, tok_budget, max_bs)
    t0, done = time.time(), 0
    for bi, b in enumerate(batches):
        safe(b)
        done += len(b)
        if (bi + 1) % log_every == 0 or bi + 1 == len(batches):
            el = time.time() - t0
            print(f"  {tag}{done}/{len(enc)} пар, {el:.0f}s, {done / el:.1f} пар/с, ETA {(len(enc) - done) / max(done / el, 1e-9):.0f}s", flush=True)
    assert np.isfinite(out).all(), "есть нечисловые скоры"
    return out, order, lens


def select_rows(tr, te, n_q):

    qs = np.random.RandomState(123).choice(tr.query_id.unique(), 400, replace=False)[:n_q]
    return np.where(tr.query_id.isin(qs).values)[0], np.where(te.query_id.isin(te.query_id.unique()[:n_q]).values)[0]


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", default="data")
    ap.add_argument("--out", default="out")
    ap.add_argument("--name", default="zs_jina1024")
    ap.add_argument("--max_len", type=int, default=1024)
    ap.add_argument("--fp16", action="store_true", help="fp16 на CUDA (на CPU/MPS игнорируется)")
    ap.add_argument("--tok_budget", type=int, default=24576)
    ap.add_argument("--max_bs", type=int, default=96)
    ap.add_argument("--shard", type=int, default=0)
    ap.add_argument("--n_shards", type=int, default=1)
    ap.add_argument("--merge", action="store_true")
    ap.add_argument("--dry_queries", type=int, default=0)
    ap.add_argument("--fp32_check", type=int, default=0, help="N пар: сверить fp16 с fp32 (только на CUDA с --fp16)")
    ap.add_argument("--folds", default=None, help="folds.npy для NDCG по фолдам")
    ap.add_argument("--log_every", type=int, default=50)
    a = ap.parse_args()
    os.makedirs(a.out, exist_ok=True)
    tr, te = load(a.data)
    n_tr, n_te = len(tr), len(te)
    queries = list(tr["query"]) + list(te["query"])
    pas = [passage(t, x) for t, x in zip(tr["title"], tr["text"])] + [passage(t, x) for t, x in zip(te["title"], te["text"])]
    if a.dry_queries:
        i_tr, i_te = select_rows(tr, te, a.dry_queries)
        rows = np.concatenate([i_tr, n_tr + i_te])
    else:
        rows = np.arange(n_tr + n_te)
    part = lambda k: os.path.join(a.out, f"{a.name}_part{k}of{a.n_shards}.npz")

    if not a.merge:
        import torch
        from transformers import AutoTokenizer
        device = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
        fp16 = a.fp16 and device.type == "cuda"
        tok = AutoTokenizer.from_pretrained(MODEL, trust_remote_code=True)
        model = get_model(fp16, device)

        enc_len = np.array([len(x) for x in tok([queries[i] for i in rows], [pas[i] for i in rows], truncation="only_second",
                                                 max_length=a.max_len, padding=False)["input_ids"]])
        mine = np.argsort(-enc_len, kind="stable")[a.shard::a.n_shards]
        sel = rows[mine]
        t0 = time.time()
        s, _, lens = score_pairs(model, tok, [queries[i] for i in sel], [pas[i] for i in sel], a.max_len, a.tok_budget, a.max_bs, device,
                                 a.log_every, tag=f"[shard {a.shard}/{a.n_shards}] ")
        info = {"fp16": fp16, "device": str(device), "sec": time.time() - t0, "n_pairs": int(len(sel)), "tok_mean": float(lens.mean()),
                "trunc_share": float((lens >= a.max_len).mean()), "torch": torch.__version__}
        try:
            import transformers
            info["transformers"] = transformers.__version__
        except Exception:
            pass
        if a.fp32_check and fp16 and a.shard == 0:
            ch = np.random.RandomState(0).choice(len(sel), min(a.fp32_check, len(sel)), replace=False)
            m32 = get_model(False, device)
            s32, _, _ = score_pairs(m32, tok, [queries[i] for i in sel[ch]], [pas[i] for i in sel[ch]], a.max_len, 8192, 32, device, 10 ** 9, tag="[fp32] ")
            info["fp16_vs_fp32"] = {"n": int(len(ch)), "pearson": float(np.corrcoef(s[ch], s32)[0, 1]), "max_abs_diff": float(np.abs(s[ch] - s32).max()),
                                    "mean_abs_diff": float(np.abs(s[ch] - s32).mean())}
            print("fp16 vs fp32:", info["fp16_vs_fp32"], flush=True)
            del m32
        np.savez(part(a.shard), rows=sel, scores=s, info=json.dumps(info))
        print(f"[shard {a.shard}] сохранено {part(a.shard)}: {info}", flush=True)
        if a.n_shards > 1:
            return

    full = np.full(n_tr + n_te, np.nan, dtype=np.float32)
    infos = []
    for k in range(a.n_shards):
        z = np.load(part(k))
        full[z["rows"]] = z["scores"]
        infos.append(json.loads(str(z["info"])))
    got = np.isfinite(full)
    assert got[rows].all(), f"не хватает {(~got[rows]).sum()} строк: не все шарды досчитаны?"
    np.save(os.path.join(a.out, f"{a.name}_train.npy"), full[:n_tr])
    np.save(os.path.join(a.out, f"{a.name}_test.npy"), full[n_tr:])
    m = got[:n_tr]
    meta = {"model": MODEL, "max_len": a.max_len, "name": a.name, "dry_queries": a.dry_queries, "n_train": n_tr, "n_test": n_te,
            "rows_done": int(got.sum()), "shards": infos}
    nd = ndcg10_per_query(tr.query_id.values[m], tr.id.values[m], tr.relevance.values[m], full[:n_tr][m])
    meta["train_ndcg10"] = float(nd.mean())
    meta["n_train_queries_scored"] = int(len(nd))
    if a.folds and os.path.exists(a.folds) and not a.dry_queries:
        folds = np.load(a.folds)
        meta["train_ndcg10_by_fold"] = [float(ndcg10_per_query(tr.query_id.values[folds == f], tr.id.values[folds == f], tr.relevance.values[folds == f],
                                                              full[:n_tr][folds == f]).mean()) for f in range(int(folds.max()) + 1)]
        meta["folds_md5"] = hashlib.md5(open(a.folds, "rb").read()).hexdigest()
    json.dump(meta, open(os.path.join(a.out, f"{a.name}_meta.json"), "w"), ensure_ascii=False, indent=1)
    print(f"сохранено {a.name}_train.npy / _test.npy; строк посчитано {int(got.sum())} из {n_tr + n_te}")
    print(json.dumps({k: v for k, v in meta.items() if k != "shards"}, ensure_ascii=False))


if __name__ == "__main__":
    main()


Writing jina/jina_zs.py


In [17]:
%%writefile jina/jina_ft.py

import argparse
import json
import os
import random
import sys
import time

import numpy as np

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import jina_zs as J


def predict(model, enc, device, amp, tok_budget=24576, max_bs=96, pad_id=1):
    import torch
    lens = np.array([len(x) for x in enc]); order = np.argsort(-lens, kind="stable")
    out = np.full(len(enc), np.nan, dtype=np.float32)
    batches = J.make_batches(lens, order, tok_budget, max_bs)
    model.eval()
    for b in batches:
        L = int(lens[b[0]])
        ids = torch.full((len(b), L), pad_id, dtype=torch.long); att = torch.zeros((len(b), L), dtype=torch.long)
        for k, j in enumerate(b):
            ids[k, :lens[j]] = torch.tensor(enc[j]); att[k, :lens[j]] = 1
        with torch.inference_mode(), torch.autocast("cuda", dtype=torch.float16, enabled=amp):
            out[b] = model(input_ids=ids.to(device), attention_mask=att.to(device), return_dict=True).logits.view(-1).float().cpu().numpy()
    model.train()
    return out


def pp_loss(s, y):
    import torch
    pos, neg = s[y > 0], s[y <= 0]
    if len(pos) == 0 or len(neg) == 0:
        return None
    lse = torch.logsumexp(torch.cat([pos[:, None], neg[None, :].expand(len(pos), -1)], 1), 1)
    return (lse - pos).mean()


def run_fold(a, f, tr, te, folds, tok, device):
    import torch
    import torch.nn.functional as F
    amp = a.fp16 and device.type == "cuda"
    random.seed(a.seed + f); np.random.seed(a.seed + f); torch.manual_seed(a.seed + f)
    tri, vai = np.where(folds != f)[0], np.where(folds == f)[0]
    te_rows = np.arange(len(te))
    if a.dry_steps:
        vq = list(dict.fromkeys(tr.query_id.values[vai]))[:3]; vai = vai[np.isin(tr.query_id.values[vai], vq)]; te_rows = te_rows[:20]
    P = lambda d, r: [J.passage(t, x) for t, x in zip(d["title"].values[r], d["text"].values[r])]
    enc = lambda d, r, L: tok(list(d["query"].values[r]), P(d, r), truncation="only_second", max_length=L, padding=False)["input_ids"]
    t0 = time.time()
    tr_ids, va_ids, te_ids = enc(tr, tri, a.train_len), enc(tr, vai, a.infer_len), enc(te, te_rows, a.infer_len)
    y = tr["relevance"].values[tri].astype(np.float32)
    groups = {}
    for i, g in enumerate(tr["query_id"].values[tri]):
        groups.setdefault(g, []).append(i)
    groups = list(groups.values())
    print(f"[fold {f}] токенизация {time.time() - t0:.0f}s; train запросов {len(groups)}, пар {len(tri)}; val {len(vai)}; test {len(te_rows)}", flush=True)

    model = J.get_model(False, device)
    emb = model.roberta.embeddings.word_embeddings
    emb.weight.requires_grad_(False)
    model.train()
    params = [p for p in model.parameters() if p.requires_grad]
    print(f"[fold {f}] обучаемых {sum(p.numel() for p in params) / 1e6:.1f}M из {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M", flush=True)
    opt = torch.optim.AdamW(params, lr=a.lr, weight_decay=0.01)
    steps_ep = (len(groups) + a.q_per_step - 1) // a.q_per_step
    total = steps_ep * a.epochs if not a.dry_steps else a.dry_steps
    warm = max(1, int(a.warmup * total))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min((s + 1) / warm, max(0.0, (total - s) / max(1, total - warm))))
    scaler = torch.amp.GradScaler("cuda", enabled=amp)
    rng = random.Random(a.seed + f)
    step, skipped, meta = 0, 0, {"fold": f, "args": vars(a), "epochs": {}}
    for ep in range(1, a.epochs + 1):
        order = list(range(len(groups))); rng.shuffle(order)
        run, nb, t0 = {"bce": 0.0, "pp": 0.0}, 0, time.time()
        opt.zero_grad(set_to_none=True)
        for bi, gi in enumerate(order):
            g = groups[gi]
            if len(g) > a.max_docs:
                pos = [i for i in g if y[i] > 0][:a.max_docs - 1]; neg = [i for i in g if y[i] <= 0]; rng.shuffle(neg); g = pos + neg[:a.max_docs - len(pos)]
            L = max(len(tr_ids[i]) for i in g)
            ids = torch.full((len(g), L), tok.pad_token_id, dtype=torch.long); att = torch.zeros((len(g), L), dtype=torch.long)
            for k, i in enumerate(g):
                ids[k, :len(tr_ids[i])] = torch.tensor(tr_ids[i]); att[k, :len(tr_ids[i])] = 1
            with torch.autocast("cuda", dtype=torch.float16, enabled=amp):
                s = model(input_ids=ids.to(device), attention_mask=att.to(device), return_dict=True).logits.view(-1).float()
            yt = torch.tensor(y[g], device=device)
            lb = F.binary_cross_entropy_with_logits(s, yt); lp = pp_loss(s, yt)
            loss = a.bce_w * lb + (lp if lp is not None else 0.0)
            scaler.scale(loss / a.q_per_step).backward()
            run["bce"] += lb.detach().item(); run["pp"] += lp.detach().item() if lp is not None else 0.0; nb += 1
            if (bi + 1) % a.q_per_step == 0 or bi + 1 == len(order):
                scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(params, 1.0)
                sc0 = scaler.get_scale() if amp else 1.0
                scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); sched.step(); step += 1
                if amp and scaler.get_scale() < sc0:
                    skipped += 1
                if step % a.log_every == 0:
                    el = time.time() - t0
                    print(f"  [fold {f}] ep{ep} шаг {step}/{total} bce {run['bce'] / nb:.4f} pp {run['pp'] / nb:.4f} lr {sched.get_last_lr()[0]:.2e} "
                          f"пропущено {skipped} {el:.0f}s" + (f" mem {torch.cuda.max_memory_allocated(device) / 2**30:.1f}GB" if device.type == "cuda" else ""), flush=True)
                    run, nb = {"bce": 0.0, "pp": 0.0}, 0
                if a.dry_steps and step >= a.dry_steps:
                    break
        tt = time.time() - t0; t1 = time.time()
        pv = predict(model, va_ids, device, amp, pad_id=tok.pad_token_id)
        nd = float(J.ndcg10_per_query(tr.query_id.values[vai], tr.id.values[vai], tr.relevance.values[vai], pv).mean())
        pt = predict(model, te_ids, device, amp, pad_id=tok.pad_token_id)
        np.save(os.path.join(a.out, f"fold{f}_ep{ep}_val.npy"), pv); np.save(os.path.join(a.out, f"fold{f}_ep{ep}_test.npy"), pt)
        meta["epochs"][ep] = {"val_ndcg10": nd, "train_sec": tt, "infer_sec": time.time() - t1, "skipped_steps": skipped,
                              "nonfinite": int((~np.isfinite(pv)).sum() + (~np.isfinite(pt)).sum())}
        print(f"[fold {f}] эпоха {ep}: val NDCG@10 {nd:.4f} (обучение {tt:.0f}s, инференс {time.time() - t1:.0f}s)", flush=True)
        if a.dry_steps:
            break
    json.dump(meta, open(os.path.join(a.out, f"fold{f}_meta.json"), "w"), ensure_ascii=False, indent=1)
    del model, opt
    if device.type == "cuda":
        torch.cuda.empty_cache()


def merge(a, tr, folds):
    res = {}
    for ep in range(1, a.epochs + 1):
        fs = [f for f in range(5) if os.path.exists(os.path.join(a.out, f"fold{f}_ep{ep}_val.npy"))
              and len(np.load(os.path.join(a.out, f"fold{f}_ep{ep}_val.npy"))) == (folds == f).sum()]
        if not fs:
            print(f"эпоха {ep}: полных фолдов нет")
            continue
        oof = np.full(len(tr), np.nan, dtype=np.float32)
        for f in fs:
            oof[folds == f] = np.load(os.path.join(a.out, f"fold{f}_ep{ep}_val.npy"))
        per = {f: float(J.ndcg10_per_query(tr.query_id.values[folds == f], tr.id.values[folds == f], tr.relevance.values[folds == f], oof[folds == f]).mean()) for f in fs}
        test = np.mean([np.load(os.path.join(a.out, f"fold{f}_ep{ep}_test.npy")) for f in fs], axis=0)
        np.save(os.path.join(a.out, f"{a.name}_ep{ep}_oof.npy"), oof); np.save(os.path.join(a.out, f"{a.name}_ep{ep}_test.npy"), test)
        res[ep] = {"folds_done": fs, "ndcg_by_fold": per, "cv_mean": float(np.mean(list(per.values())))}
        print(f"эпоха {ep}: фолды {fs}, NDCG по фолдам {[round(v, 4) for v in per.values()]}, среднее {res[ep]['cv_mean']:.4f}")
    json.dump(res, open(os.path.join(a.out, f"{a.name}_meta.json"), "w"), ensure_ascii=False, indent=1)
    print("для сравнения zero-shot Jina по фолдам: 0.9540 0.9560 0.9555 0.9610 0.9538 (среднее 0.9561)")


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", default="data")
    ap.add_argument("--out", default="ft_out")
    ap.add_argument("--name", default="ft_jina")
    ap.add_argument("--folds", required=True)
    ap.add_argument("--run_folds", type=int, nargs="*", default=[0, 1, 2, 3, 4])
    ap.add_argument("--epochs", type=int, default=2)
    ap.add_argument("--lr", type=float, default=2e-5)
    ap.add_argument("--warmup", type=float, default=0.1)
    ap.add_argument("--train_len", type=int, default=512)
    ap.add_argument("--infer_len", type=int, default=1024)
    ap.add_argument("--q_per_step", type=int, default=8)
    ap.add_argument("--max_docs", type=int, default=16)
    ap.add_argument("--bce_w", type=float, default=0.5)
    ap.add_argument("--seed", type=int, default=42)
    ap.add_argument("--fp16", action="store_true")
    ap.add_argument("--merge", action="store_true")
    ap.add_argument("--dry_steps", type=int, default=0)
    ap.add_argument("--log_every", type=int, default=25)
    a = ap.parse_args()
    os.makedirs(a.out, exist_ok=True)
    tr, te = J.load(a.data)
    folds = np.load(a.folds); assert len(folds) == len(tr)
    if a.merge:
        return merge(a, tr, folds)
    import torch
    from transformers import AutoTokenizer
    device = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
    tok = AutoTokenizer.from_pretrained(J.MODEL, trust_remote_code=True)
    print(f"device {device}, fp16 {a.fp16 and device.type == 'cuda'}, torch {torch.__version__}, фолды {a.run_folds}", flush=True)
    for f in a.run_folds:
        if os.path.exists(os.path.join(a.out, f"fold{f}_meta.json")) and not a.dry_steps:
            print(f"[fold {f}] уже готов, пропуск"); continue
        run_fold(a, f, tr, te, folds, tok, device)


if __name__ == "__main__":
    main()


Writing jina/jina_ft.py


In [18]:
%%writefile jina/jina_v3_zs.py

import argparse
import hashlib
import json
import os
import sys
import time

import numpy as np
import pandas as pd

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import jina_zs as J


def get_model(name, fp16, device):
    import torch
    from transformers import AutoModel, AutoTokenizer
    try:
        m = AutoModel.from_pretrained(name, trust_remote_code=True, dtype=torch.float32, attn_implementation="sdpa")
    except TypeError:
        m = AutoModel.from_pretrained(name, trust_remote_code=True, torch_dtype=torch.float32, attn_implementation="sdpa")
    print("attn_implementation:", getattr(m.config, "_attn_implementation", "?"), flush=True)
    m = (m.half() if fp16 else m).to(device).eval()
    tok = AutoTokenizer.from_pretrained(name, trust_remote_code=True)
    fmt = sys.modules[type(m).__module__].format_docs_prompts_func
    return m, tok, fmt


def truncate(tok, text, max_tokens):
    ids = tok(text, add_special_tokens=False)["input_ids"]
    return text if len(ids) <= max_tokens else tok.decode(ids[:max_tokens])


def _scores_block(m, tok, fmt, q, docs, device, stats):

    import torch
    prompt = fmt(q, docs, instruction=None, special_tokens=m.special_tokens, no_thinking=True)
    enc = tok(text=[prompt], return_tensors="pt").to(device)
    try:
        with torch.inference_mode():
            s = m(**enc).scores.view(-1).float().cpu().numpy()
        stats["tok"] += enc["input_ids"].shape[1]
        return s
    except torch.OutOfMemoryError:
        del enc
        torch.cuda.empty_cache()
        if len(docs) == 1:
            raise
        stats["splits"] += 1
        h = len(docs) // 2
        return np.concatenate([_scores_block(m, tok, fmt, q, docs[:h], device, stats), _scores_block(m, tok, fmt, q, docs[h:], device, stats)])


def score_queries(m, tok, fmt, groups, device, max_doc_tokens, log_every=200, tag=""):

    out, t0, stats = [], time.time(), {"tok": 0, "splits": 0}
    for gi, (q, docs) in enumerate(groups):
        docs = [truncate(tok, d, max_doc_tokens) for d in docs]
        res = []
        for perm in (np.arange(len(docs)), np.arange(len(docs))[::-1]):
            s = _scores_block(m, tok, fmt, q, [docs[i] for i in perm], device, stats)
            assert len(s) == len(docs), (len(s), len(docs))
            back = np.empty(len(docs), dtype=np.float32); back[perm] = s
            res.append(back)
        out.append(np.stack(res))
        if (gi + 1) % log_every == 0 or gi + 1 == len(groups):
            el = time.time() - t0
            print(f"  {tag}{gi + 1}/{len(groups)} запросов, {el:.0f}s, {stats['tok'] / el:.0f} ток/с, делений по OOM {stats['splits']}, "
                  f"ETA {(len(groups) - gi - 1) * el / (gi + 1):.0f}s", flush=True)
    return out


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", default="data")
    ap.add_argument("--out", default="out")
    ap.add_argument("--model", default="jinaai/jina-reranker-v3.5")
    ap.add_argument("--name", default="zs_jv35")
    ap.add_argument("--max_doc_tokens", type=int, default=1024)
    ap.add_argument("--fp16", action="store_true")
    ap.add_argument("--fp32_check", type=int, default=0, help="N запросов: сверить fp16 с fp32 (CUDA)")
    ap.add_argument("--shard", type=int, default=0)
    ap.add_argument("--n_shards", type=int, default=1)
    ap.add_argument("--merge", action="store_true")
    ap.add_argument("--dry_queries", type=int, default=0)
    ap.add_argument("--folds", default=None)
    ap.add_argument("--log_every", type=int, default=200)
    a = ap.parse_args()
    os.makedirs(a.out, exist_ok=True)
    tr, te = J.load(a.data)
    n_tr = len(tr)
    df = pd.concat([tr[["query_id", "query", "title", "text"]], te[["query_id", "query", "title", "text"]]], ignore_index=True)
    df["passage"] = [J.passage(t, x) for t, x in zip(df["title"], df["text"])]
    rows_all = np.arange(len(df))
    if a.dry_queries:
        i_tr, i_te = J.select_rows(tr, te, a.dry_queries)
        rows_all = np.concatenate([i_tr, n_tr + i_te])
    sub = df.iloc[rows_all]
    qorder = list(dict.fromkeys(sub["query_id"]))
    qrows = sub.groupby("query_id", sort=False).apply(lambda g: g.index.values, include_groups=False).to_dict()
    qlen = np.array([sub.loc[qrows[q], "passage"].str.len().sum() for q in qorder])
    part = lambda k: os.path.join(a.out, f"{a.name}_part{k}of{a.n_shards}.npz")

    if not a.merge:
        import torch
        device = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
        fp16 = a.fp16 and device.type == "cuda"
        mine = [qorder[i] for i in np.argsort(-qlen, kind="stable")[a.shard::a.n_shards]]
        groups = [(sub.loc[qrows[q], "query"].iloc[0], list(sub.loc[qrows[q], "passage"])) for q in mine]
        m, tok, fmt = get_model(a.model, fp16, device)
        t0 = time.time()
        sc = score_queries(m, tok, fmt, groups, device, a.max_doc_tokens, a.log_every, tag=f"[{a.name} shard {a.shard}] ")
        rows = np.concatenate([qrows[q] for q in mine]); s2 = np.concatenate(sc, axis=1)
        import transformers
        info = {"fp16": fp16, "device": str(device), "sec": time.time() - t0, "n_queries": len(mine), "n_pairs": int(len(rows)),
                "torch": torch.__version__, "transformers": transformers.__version__}
        if a.fp32_check and fp16 and a.shard == 0:
            mid = len(groups) // 2
            ch = list(range(mid, min(len(groups), mid + a.fp32_check)))
            del m; torch.cuda.empty_cache()
            m32, _, _ = get_model(a.model, False, device)
            s32 = np.concatenate(score_queries(m32, tok, fmt, [groups[i] for i in ch], device, a.max_doc_tokens, 10 ** 9, tag="[fp32] "), axis=1)
            s16 = np.concatenate([sc[i] for i in ch], axis=1)
            info["fp16_vs_fp32"] = {"n_pairs": int(s32.shape[1]), "pearson": float(np.corrcoef(s16.ravel(), s32.ravel())[0, 1]),
                                    "max_abs_diff": float(np.abs(s16 - s32).max())}
            print("fp16 vs fp32:", info["fp16_vs_fp32"], flush=True)
        np.savez(part(a.shard), rows=rows, s=s2, info=json.dumps(info))
        print(f"[{a.name} shard {a.shard}] сохранено: {info}", flush=True)
        if a.n_shards > 1:
            return

    full = np.full((2, len(df)), np.nan, dtype=np.float32); infos = []
    for k in range(a.n_shards):
        z = np.load(part(k)); full[:, z["rows"]] = z["s"]; infos.append(json.loads(str(z["info"])))
    assert np.isfinite(full[:, rows_all]).all(), "не все строки посчитаны"
    avg = full.mean(0)
    np.save(os.path.join(a.out, f"{a.name}_train.npy"), avg[:n_tr]); np.save(os.path.join(a.out, f"{a.name}_test.npy"), avg[n_tr:])
    np.save(os.path.join(a.out, f"{a.name}_perm2_train.npy"), full[:, :n_tr]); np.save(os.path.join(a.out, f"{a.name}_perm2_test.npy"), full[:, n_tr:])
    mtr = np.isfinite(avg[:n_tr])
    nd = lambda v, mm: float(J.ndcg10_per_query(tr.query_id.values[mm], tr.id.values[mm], tr.relevance.values[mm], v[mm]).mean())
    meta = {"model": a.model, "name": a.name, "max_doc_tokens": a.max_doc_tokens, "dry_queries": a.dry_queries, "shards": infos,
            "train_ndcg10": nd(avg[:n_tr], mtr), "train_ndcg10_order_csv": nd(full[0, :n_tr], mtr), "train_ndcg10_order_reversed": nd(full[1, :n_tr], mtr),
            "corr_two_orders": float(np.corrcoef(full[0, rows_all], full[1, rows_all])[0, 1])}
    if a.folds and os.path.exists(a.folds) and not a.dry_queries:
        fo = np.load(a.folds)
        meta["train_ndcg10_by_fold"] = [nd(avg[:n_tr], fo == f) for f in range(int(fo.max()) + 1)]
        meta["folds_md5"] = hashlib.md5(open(a.folds, "rb").read()).hexdigest()
    json.dump(meta, open(os.path.join(a.out, f"{a.name}_meta.json"), "w"), ensure_ascii=False, indent=1)
    print(json.dumps({k: v for k, v in meta.items() if k != "shards"}, ensure_ascii=False))


if __name__ == "__main__":
    main()


Writing jina/jina_v3_zs.py


In [19]:
%%writefile jina/jina_v3_ft.py


import argparse
import json
import math
import os
import random
import sys
import time

import numpy as np

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import jina_zs as J

DOC_ID, Q_ID = 151670, 151671
TARGETS = ("q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj")


def add_lora(m, r, alpha, dropout):
    import torch
    from torch import nn

    class LoRALinear(nn.Module):
        def __init__(self, base):
            super().__init__()
            self.base = base
            self.A = nn.Parameter(torch.empty(r, base.in_features, dtype=torch.float32, device=base.weight.device))
            self.B = nn.Parameter(torch.zeros(base.out_features, r, dtype=torch.float32, device=base.weight.device))
            nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
            self.drop = nn.Dropout(dropout); self.scale = alpha / r

        def forward(self, x):
            return self.base(x) + (self.drop(x) @ self.A.t().to(x.dtype)) @ self.B.t().to(x.dtype) * self.scale

    n = 0
    for name, mod in list(m.model.named_modules()):
        for t in TARGETS:
            child = getattr(mod, t, None)
            if isinstance(child, nn.Linear):
                setattr(mod, t, LoRALinear(child)); n += 1
    return n


def get_model(name, fp16, device, lora_r=0, lora_alpha=32, lora_dropout=0.05):
    import torch
    from transformers import AutoModel, AutoTokenizer
    try:
        m = AutoModel.from_pretrained(name, trust_remote_code=True, dtype=torch.float32, attn_implementation="sdpa")
    except TypeError:
        m = AutoModel.from_pretrained(name, trust_remote_code=True, torch_dtype=torch.float32, attn_implementation="sdpa")
    print("attn_implementation:", getattr(m.config, "_attn_implementation", "?"), flush=True)
    if fp16:
        m.model.half()
    m = m.to(device)
    for p in m.parameters():
        p.requires_grad_(False)
    n = add_lora(m, lora_r, lora_alpha, lora_dropout) if lora_r else 0
    for p in m.projector.parameters():
        p.float(); p.requires_grad_(True)
    m.projector.float()
    tok = AutoTokenizer.from_pretrained(name, trust_remote_code=True)
    fmt = sys.modules[type(m).__module__].format_docs_prompts_func
    print(f"LoRA-слоёв {n}, обучаемых параметров {sum(p.numel() for p in m.parameters() if p.requires_grad) / 1e6:.2f}M", flush=True)
    return m, tok, fmt


def doc_scores(m, ids):

    import torch.nn.functional as F
    h = m.model(input_ids=ids).last_hidden_state[0]
    d = m.projector(h[ids[0] == DOC_ID].to(m.projector[0].weight.dtype))
    q = m.projector(h[ids[0] == Q_ID].to(m.projector[0].weight.dtype))
    return F.cosine_similarity(d.float(), q.float().expand_as(d), dim=-1)


def encode(m, tok, fmt, q, docs, device):
    prompt = fmt(q, docs, instruction=None, special_tokens=m.special_tokens, no_thinking=True)
    return tok(text=[prompt], return_tensors="pt")["input_ids"].to(device)


def truncate_all(tok, texts, max_tokens):
    out = []
    for i in range(0, len(texts), 512):
        chunk = texts[i:i + 512]
        ids = tok(chunk, add_special_tokens=False)["input_ids"]
        out += [t if len(x) <= max_tokens else tok.decode(x[:max_tokens]) for t, x in zip(chunk, ids)]
    return out


def infer(m, tok, fmt, groups, device, amp, tag="", log_every=100):
    import torch

    def block(q, docs):
        ids = encode(m, tok, fmt, q, docs, device)
        try:
            with torch.inference_mode(), torch.autocast("cuda", dtype=torch.float16, enabled=amp):
                s = doc_scores(m, ids).cpu().numpy()
            st["tok"] += ids.shape[1]
            assert len(s) == len(docs)
            return s
        except torch.OutOfMemoryError:
            del ids; torch.cuda.empty_cache()
            if len(docs) == 1:
                raise
            st["splits"] += 1; h = len(docs) // 2
            return np.concatenate([block(q, docs[:h]), block(q, docs[h:])])

    m.eval(); out = []; t0 = time.time(); st = {"tok": 0, "splits": 0}
    for gi, (q, docs) in enumerate(groups):
        out.append(block(q, docs).astype(np.float32))
        if (gi + 1) % log_every == 0 or gi + 1 == len(groups):
            el = time.time() - t0
            print(f"  {tag}{gi + 1}/{len(groups)}, {el:.0f}s, {st['tok'] / max(el, 1e-9):.0f} ток/с, делений по OOM {st['splits']}", flush=True)
    m.train()
    return out


def pp_loss(s, y):
    import torch
    pos, neg = s[y > 0], s[y <= 0]
    lse = torch.logsumexp(torch.cat([pos[:, None], neg[None, :].expand(len(pos), -1)], 1), 1)
    return (lse - pos).mean()


def groups_of(df, rows, passages):

    qid = df["query_id"].values[rows]
    order = list(dict.fromkeys(qid))
    pos = {}
    for k, q in enumerate(qid):
        pos.setdefault(q, []).append(rows[k])
    idx = [np.array(pos[q]) for q in order]
    return order, [(df["query"].values[i[0]], [passages[j] for j in i]) for i in idx], idx


def run_fold(a, f, tr, te, folds, device):
    import torch
    amp = a.fp16 and device.type == "cuda"
    random.seed(a.seed + f); np.random.seed(a.seed + f); torch.manual_seed(a.seed + f)
    m, tok, fmt = get_model(a.model, amp, device, a.lora_r, a.lora_alpha, a.lora_dropout)
    P_tr = [J.passage(t, x) for t, x in zip(tr["title"].values, tr["text"].values)]
    P_te = [J.passage(t, x) for t, x in zip(te["title"].values, te["text"].values)]
    tri, vai = np.where(folds != f)[0], np.where(folds == f)[0]
    t0 = time.time()
    P_tr_short = truncate_all(tok, P_tr, a.train_doc_tokens)
    _, tr_groups, tr_idx = groups_of(tr, tri, P_tr_short)
    rel = tr["relevance"].values
    print(f"[fold {f}] train запросов {len(tr_groups)}, обрезка {time.time() - t0:.0f}s", flush=True)

    lora_p = [p for n, p in m.named_parameters() if p.requires_grad and not n.startswith("projector")]
    proj_p = list(m.projector.parameters())
    opt = torch.optim.AdamW([{"params": lora_p, "lr": a.lr}, {"params": proj_p, "lr": a.lr * a.proj_lr_mult}], weight_decay=0.0)
    base_lrs = [g["lr"] for g in opt.param_groups]
    scaler = torch.amp.GradScaler("cuda", enabled=amp)
    if a.grad_ckpt:
        m.model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    m.train()
    budget = a.train_minutes * 60; nq_total = a.max_epochs * len(tr_groups)
    t0 = time.time(); done = 0; steps = 0; ooms = 0; skipped = 0; tok_sum = 0; loss_hist = []
    rng = np.random.RandomState(a.seed + f)
    progress = lambda: max((time.time() - t0) / budget, done / nq_total)
    stop = False
    while not stop:
        for qi in rng.permutation(len(tr_groups)):
            if progress() >= 1 or (a.dry_steps and steps >= a.dry_steps):
                stop = True; break
            q, docs = tr_groups[qi]; ridx = tr_idx[qi]; y = rel[ridx]
            P, N = np.where(y > 0)[0], np.where(y <= 0)[0]
            done += 1
            if len(P) == 0 or len(N) == 0:
                skipped += 1; continue
            P = rng.permutation(P)[:a.max_pos]; N = rng.permutation(N)[:max(1, a.max_docs - len(P))]
            sel = rng.permutation(np.concatenate([P, N]))
            ids = encode(m, tok, fmt, q, [docs[i] for i in sel], device)
            yy = torch.tensor(y[sel], device=device)
            try:
                with torch.autocast("cuda", dtype=torch.float16, enabled=amp):
                    s = doc_scores(m, ids)
                loss = pp_loss(s / a.tau, yy) / a.accum
                scaler.scale(loss).backward()
                loss_hist.append(loss.item() * a.accum); tok_sum += ids.shape[1]
            except torch.OutOfMemoryError:
                ooms += 1; del ids; opt.zero_grad(set_to_none=True); torch.cuda.empty_cache()
                if not getattr(m.model, "gradient_checkpointing", False):
                    m.model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
                    print(f"  [fold {f}] OOM -> включён gradient checkpointing", flush=True)
                continue
            if done % a.accum == 0:
                p = progress(); w = min(1.0, p / 0.05) * max(0.0, 1.0 - p)
                for g, lr0 in zip(opt.param_groups, base_lrs):
                    g["lr"] = lr0 * w
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_([x for g in opt.param_groups for x in g["params"]], 1.0)
                scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); steps += 1
                if steps % a.log_every == 0 or a.dry_steps:
                    el = time.time() - t0
                    print(f"  [fold {f}] шаг {steps}, запросов {done} ({done / len(tr_groups):.2f} эп.), loss {np.mean(loss_hist[-a.log_every * a.accum:]):.4f}, "
                          f"lr x{w:.2f}, {el:.0f}s, {tok_sum / el:.0f} ток/с, OOM {ooms}, прогресс {p:.2f}", flush=True)
    train_info = {"sec": time.time() - t0, "queries_seen": done, "epochs": done / len(tr_groups), "steps": steps, "ooms": ooms, "skipped": skipped,
                  "grad_ckpt": bool(getattr(m.model, "gradient_checkpointing", False)), "loss_first100": float(np.mean(loss_hist[:100])) if loss_hist else None, "loss_last100": float(np.mean(loss_hist[-100:])) if loss_hist else None}
    print(f"[fold {f}] обучение: {train_info}", flush=True)
    torch.cuda.empty_cache()

    P_tr_inf = truncate_all(tok, [P_tr[i] for i in vai], a.infer_doc_tokens)
    full = {i: p for i, p in zip(vai, P_tr_inf)}
    _, v_groups, v_idx = groups_of(tr, vai, full)
    if a.dry_steps:
        v_groups, v_idx = v_groups[:3], v_idx[:3]
    vs = infer(m, tok, fmt, v_groups, device, amp, tag=f"[fold {f} val] ")
    val = np.full(len(tr), np.nan, dtype=np.float32)
    for i, s in zip(v_idx, vs):
        val[i] = s
    mv = np.isfinite(val)
    nd = J.ndcg10_per_query(tr.query_id.values[mv], tr.id.values[mv], tr.relevance.values[mv], val[mv]).mean()
    print(f"[fold {f}] NDCG@10 val {nd:.4f} (zero-shot v3 на этом фолде: один порядок ~ см. meta zs)", flush=True)
    np.save(os.path.join(a.out, f"fold{f}_val.npy"), val)
    info = {"fold": f, "val_ndcg10": float(nd), "train": train_info, "args": vars(a)}
    if f in a.test_folds:
        te_rows = np.arange(len(te))
        full = dict(zip(te_rows, truncate_all(tok, P_te, a.infer_doc_tokens)))
        _, t_groups, t_idx = groups_of(te, te_rows, full)
        if a.dry_steps:
            t_groups, t_idx = t_groups[:2], t_idx[:2]
        ts = infer(m, tok, fmt, t_groups, device, amp, tag=f"[fold {f} test] ", log_every=200)
        test = np.full(len(te), np.nan, dtype=np.float32)
        for i, s in zip(t_idx, ts):
            test[i] = s
        np.save(os.path.join(a.out, f"fold{f}_test.npy"), test)
        info["test_done"] = True
    json.dump(info, open(os.path.join(a.out, f"fold{f}_info.json"), "w"), ensure_ascii=False, indent=1)
    del m; torch.cuda.empty_cache()


def merge(a, tr, te, folds):
    import glob
    oof = np.full(len(tr), np.nan, dtype=np.float32); done = []; tests = []
    for f in range(int(folds.max()) + 1):
        p = os.path.join(a.out, f"fold{f}_val.npy")
        if not os.path.exists(p):
            continue
        v = np.load(p); mk = folds == f
        if not np.isfinite(v[mk]).all():
            print(f"фолд {f}: неполный, пропускаю"); continue
        oof[mk] = v[mk]; done.append(f)
    for p in sorted(glob.glob(os.path.join(a.out, "fold*_test.npy"))):
        t = np.load(p)
        if np.isfinite(t).all():
            tests.append(t)
    mo = np.isfinite(oof)
    nd = lambda mm: float(J.ndcg10_per_query(tr.query_id.values[mm], tr.id.values[mm], tr.relevance.values[mm], oof[mm]).mean())
    meta = {"folds_done": done, "n_test_models": len(tests), "oof_ndcg10": nd(mo) if mo.any() else None,
            "by_fold": {f: nd(folds == f) for f in done}}
    np.save(os.path.join(a.out, "ft_jv3_oof.npy"), oof)
    if tests:
        np.save(os.path.join(a.out, "ft_jv3_test.npy"), np.mean(tests, 0))
    json.dump(meta, open(os.path.join(a.out, "ft_jv3_meta.json"), "w"), ensure_ascii=False, indent=1)
    print(json.dumps(meta, ensure_ascii=False))


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", default="data"); ap.add_argument("--out", default="out"); ap.add_argument("--folds", required=True)
    ap.add_argument("--model", default="jinaai/jina-reranker-v3")
    ap.add_argument("--run_folds", type=int, nargs="*", default=[]); ap.add_argument("--test_folds", type=int, nargs="*", default=[])
    ap.add_argument("--train_minutes", type=float, default=75); ap.add_argument("--max_epochs", type=float, default=2.0)
    ap.add_argument("--lr", type=float, default=1e-4); ap.add_argument("--proj_lr_mult", type=float, default=0.2)
    ap.add_argument("--lora_r", type=int, default=16); ap.add_argument("--lora_alpha", type=float, default=32); ap.add_argument("--lora_dropout", type=float, default=0.05)
    ap.add_argument("--tau", type=float, default=0.05); ap.add_argument("--accum", type=int, default=8)
    ap.add_argument("--max_docs", type=int, default=8); ap.add_argument("--max_pos", type=int, default=4)
    ap.add_argument("--train_doc_tokens", type=int, default=256); ap.add_argument("--infer_doc_tokens", type=int, default=512)
    ap.add_argument("--fp16", action="store_true"); ap.add_argument("--grad_ckpt", action="store_true"); ap.add_argument("--seed", type=int, default=42)
    ap.add_argument("--dry_steps", type=int, default=0); ap.add_argument("--log_every", type=int, default=20); ap.add_argument("--merge", action="store_true")
    a = ap.parse_args()
    os.makedirs(a.out, exist_ok=True)
    tr, te = J.load(a.data)
    folds = np.load(a.folds); assert len(folds) == len(tr)
    if a.merge:
        return merge(a, tr, te, folds)
    import torch
    device = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
    for f in a.run_folds:
        run_fold(a, f, tr, te, folds, device)


if __name__ == "__main__":
    main()


Writing jina/jina_v3_ft.py


## 1. Фолды


In [20]:
FOLDS_B64 = "..."
import hashlib
raw = gzip.decompress(base64.b64decode(FOLDS_B64)); assert hashlib.md5(raw).hexdigest() == "ade68c0274ded21d5989a1e551c8ce06"
open(f"{ART}/folds.npy", "wb").write(raw)
folds = np.load(f"{ART}/folds.npy"); tr = pd.read_csv(f"{DATA}/train.csv"); assert len(folds) == len(tr)
print("строк по фолдам:", np.bincount(folds), "| запросов:", [tr.query_id[folds == f].nunique() for f in range(5)])

строк по фолдам: [6307 6306 6305 6306 6306] | запросов: [607, 610, 607, 610, 608]


## 2. Препроцессинг и признаки


In [21]:
%pip install pymorphy3==2.0.6 pymorphy3-dicts-ru

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 61.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [pymorphy3]


In [ ]:
if RUN["features"] and not done(f"{ART}/feats_train.pkl", f"{ART}/feats_test.pkl"):
    sh([PY, "-m", "src.features", "--data", DATA, "--out", ART, "--emb_models",
        "intfloat/multilingual-e5-small", "intfloat/multilingual-e5-base", "BAAI/bge-m3"])

$ /usr/bin/python3 -m src.features --data /kaggle/input/competitions/amml-ranking-competition-2026 --out artifacts --emb_models intfloat/multilingual-e5-small intfloat/multilingual-e5-base BAAI/bge-m3
лемматизация...
tf-idf...
эмбеддинги intfloat/multilingual-e5-small


эмбеддинги intfloat/multilingual-e5-base


эмбеддинги BAAI/bge-m3



признаков: 83; CV NDCG@10 одиночных бейзлайнов (среднее по фолдам | по всем запросам):
  random                           0.5683 ± 0.0084 | 0.5683
  -passage_index                   0.6470 ± 0.0119 | 0.6470
  bm25_text                        0.6851 ± 0.0021 | 0.6851
  bm25_title                       0.6626 ± 0.0084 | 0.6626
  bm25_full                        0.7146 ± 0.0028 | 0.7146
  tfidf_word                       0.7023 ± 0.0064 | 0.7023
  tfidf_char                       0.7116 ± 0.0051 | 0.7116
  q_idfcov_text                    0.6841 ± 0.0058 | 0.6841
  emb_multilingual_e5_small        0.8371 ± 0.0032 | 0.8371
  emb_multilingual_e5_base       

## 3. Кросс-энкодер bge-reranker-v2-m3 — дообучение


In [ ]:
CE_TAG = "bge-reranker-v2-m3_len320_s42"
if RUN["ce"] and not done(f"{ART}/ce_{CE_TAG}/oof.npy"):
    sh([PY, "-m", "src.train_ce", "--data", DATA, "--out", ART, "--model", "BAAI/bge-reranker-v2-m3", "--epochs", 2, "--max_len", 320,
        "--lr", "1e-5", "--q_per_batch", 1, "--grad_accum", 4, "--freeze_emb", "--grad_ckpt", "--max_docs_step", 12, "--fp16",
        "--loss", "bce+list", "--seed", 42, "--log_every", 500])

$ /usr/bin/python3 -m src.train_ce --data /kaggle/input/competitions/amml-ranking-competition-2026 --out artifacts --model BAAI/bge-reranker-v2-m3 --epochs 2 --max_len 320 --lr 1e-5 --q_per_batch 1 --grad_accum 4 --freeze_emb --grad_ckpt --max_docs_step 12 --fp16 --loss bce+list --seed 42 --log_every 500
device=cuda amp=True tag=bge-reranker-v2-m3_len320_s42
folds: загружены из artifacts/folds.npy
токенов на пару: mean 205, p90 320, обрезано до max_len: 14.8%

[fold 0] параметров 568M, обучаемых 312M; веса+град+Adam ≈ 5.6 ГБ (+ активации)
[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
[fold 0] ep1 500/2435 loss 0.8368 333s
[fold 0] ep1 1000/2435 loss 0.7604 666s
[fold 0] ep1 1500/2435 loss 0.7483 1001s
[fold 0] ep1 2000/2435 loss 0.7344 1334s
[fold 0] ep1 loss 0.7203 time 1626s peak_mem 6.8GB
[fold 0] ep2 500/2435 loss 0.3821 1961s
[fold 0] ep2 1000/2435 loss 0.3707 2291s
[fold 0] ep2 1500/2435 loss 0.3747 2628s
[fold 0] ep2 2000

## 4. Zero-shot скоры для стека (признаки, без обучения)


In [ ]:
NS = max(1, min(NGPU, 2))
if RUN["zs_bge"] and not done(f"{ART}/zs_rr1024_train.npy"):
    base = [PY, "-u", "-m", "src.zs_rerank", "--data", DATA, "--out", ART, "--model", "BAAI/bge-reranker-v2-m3", "--name", "zs_rr1024",
            "--max_len", 1024, "--fp16", "--tok_budget", 32768, "--n_shards", NS]
    par([base + ["--shard", k] for k in range(NS)], [f"{ART}/zs_rr1024_s{k}.log" for k in range(NS)]); sh(base + ["--merge"])
if RUN["zs_qwen"] and not done(f"{ART}/qwen4b_train.npy"):
    base = [PY, "-u", "-m", "src.llm_zs_all", "--data", DATA, "--out", ART, "--model", "Qwen/Qwen3-Reranker-4B", "--name", "qwen4b",
            "--dtype", "fp16", "--max_len", 768, "--tok_budget", 16384, "--chunk", 2000, "--n_shards", NS]
    par([base + ["--shard", k] for k in range(NS)], [f"{ART}/qwen4b_s{k}.log" for k in range(NS)]); sh(base + ["--merge"])

[GPU0] $ /usr/bin/python3 -u -m src.llm_zs_all --data /kaggle/input/competitions/amml-ranking-competition-2026 --out artifacts --model Qwen/Qwen3-Reranker-4B --name qwen4b --dtype fp16 --max_len 768 --tok_budget 16384 --chunk 2000 --n_shards 2 --shard 0
[GPU1] $ /usr/bin/python3 -u -m src.llm_zs_all --data /kaggle/input/competitions/amml-ranking-competition-2026 --out artifacts --model Qwen/Qwen3-Reranker-4B --name qwen4b --dtype fp16 --max_len 768 --tok_budget 16384 --chunk 2000 --n_shards 2 --shard 1
[artifacts/qwen4b_s1.log] Loading weights:  52%|█████▏    | 208/398 [00:05<00:05, 33.22it/s]
[artifacts/qwen4b_s0.log]   веса загружены за 66s
[artifacts/qwen4b_s1.log]   веса загружены за 66s
[artifacts/qwen4b_s0.log]   веса загружены за 66s
[artifacts/qwen4b_s1.log]   веса загружены за 66s
[artifacts/qwen4b_s0.log]   веса загружены за 66s
[artifacts/qwen4b_s1.log]   веса загружены за 66s
[artifacts/qwen4b_s0.log]   веса загружены за 66s
[artifacts/qwen4b_s1.log]   веса загружены за 66s

## 5. Стекинг: LightGBM LambdaRank


In [ ]:
CE_TAG = "bge-reranker-v2-m3_len320_s42"

In [ ]:
if RUN["stack"] and not done(f"{ART}/stack_bge5_qwen4b_zs_nomix_oof.npy"):
    sh([PY, "-m", "src.stack", "--out", ART, "--ce", CE_TAG, "--seeds", 5, "--tag", "bge5_qwen4b_zs_nomix",
        "--extra_feat", f"{ART}/qwen4b_train.npy", "qwen4b", "--extra_feat", f"{ART}/zs_rr1024_train.npy", "zs_rr1024"])

$ /usr/bin/python3 -m src.stack --out artifacts --ce bge-reranker-v2-m3_len320_s42 --seeds 5 --tag bge5_qwen4b_zs_nomix --extra_feat artifacts/qwen4b_train.npy qwen4b --extra_feat artifacts/zs_rr1024_train.npy zs_rr1024
внешний признак qwen4b: artifacts/qwen4b_train.npy (+ _test), строк train со скором 31530 из 31530
внешний признак zs_rr1024: artifacts/zs_rr1024_train.npy (+ _test), строк train со скором 31530 из 31530
признаков: 97; CE: ['ce_bge_reranker_v2_m3_len320_s42']
  одиночный ce_bge_reranker_v2_m3_len320_s42: 0.9368 ± 0.0023
  одиночный внешний qwen4b: 0.9394 ± 0.0043
  одиночный внешний zs_rr1024: 0.9243 ± 0.0047
лучшее число раундов (сглаж. средняя кривая): 261; lgb-ndcg@10 там 0.9530
СТЕК [bge5_qwen4b_zs_nomix] CV NDCG@10: 0.9535 ± 0.0019 (фолды [0.9534, 0.9522, 0.9523, 0.9572, 0.9526]), по всем запросам 0.9535
топ-15 признаков по gain: qwen4b_z, ce_bge_reranker_v2_m3_len320_s42_z, qwen4b_dmax, qwen4b, ce_bge_reranker_v2_m3_len320_s42, qwen4b_rank, ce_bge_reranker_v2_m3_l

## 6. Qwen3-Reranker-4B + LoRA — дообучение


In [ ]:
for vf, seed in [(0, 42), (1, 43)]:
    tag = "qwen4b_lora_t4" + ("" if vf == 0 else "_f1")
    if RUN["lora"] and not done(f"{ART}/ce_{tag}/test.npy"):
        sh([PY, "-m", "torch.distributed.run", "--nproc_per_node", max(NGPU, 1), "--master_addr", "127.0.0.1", "--master_port", 29533,
            "-m", "src.train_lora_t4", "--data", DATA, "--out", ART, "--model", "Qwen/Qwen3-Reranker-4B", "--tag", tag,
            "--val_fold", vf, "--seed", seed, "--train_len", 512, "--infer_len", 768, "--max_docs", 10, "--q_per_step", 8,
            "--lr", "1e-4", "--warmup", 0.05, "--lora_r", 16, "--lora_alpha", 32, "--lora_dropout", 0.05,
            "--loss", "pp+bce", "--bce_w", 0.5, "--tok_budget", 16384, "--ckpt_every", 25])

$ /usr/bin/python3 -m torch.distributed.run --nproc_per_node 2 --master_addr 127.0.0.1 --master_port 29533 -m src.train_lora_t4 --data /kaggle/input/competitions/amml-ranking-competition-2026 --out artifacts --model Qwen/Qwen3-Reranker-4B --tag qwen4b_lora_t4 --val_fold 0 --seed 42 --train_len 512 --infer_len 768 --max_docs 10 --q_per_step 8 --lr 1e-4 --warmup 0.05 --lora_r 16 --lora_alpha 32 --lora_dropout 0.05 --loss pp+bce --bce_w 0.5 --tok_budget 16384 --ckpt_every 25

*****************************************
Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
*****************************************
[compat] torchao 0.10.0 < 0.16.0 несовместим с peft: скрыт для этого процесса
[compat] torchao 0.10.0 < 0.16.0 несовместим с peft: скрыт для этого процесса



  обучаемых 33.0M (dtype torch.float32), база torch.float16
  продолж

## 7. Jina reranker v2 — zero-shot и полное дообучение


In [24]:
J2 = f"{ART}/jina_v2"; os.makedirs(J2, exist_ok=True)
if RUN["jina_v2_zs"] and not done(f"{J2}/zs_jina1024_train.npy"):
    base = [PY, "-u", "jina/jina_zs.py", "--data", DATA, "--out", J2, "--name", "zs_jina1024", "--max_len", 1024, "--fp16",
            "--n_shards", NS, "--folds", f"{ART}/folds.npy"]
    par([base + ["--shard", k] for k in range(NS)], [f"{J2}/zs_s{k}.log" for k in range(NS)]); sh(base + ["--merge"])
JF = f"{ART}/jina_v2_ft"; os.makedirs(JF, exist_ok=True)
if RUN["jina_v2_ft"] and not done(f"{JF}/ft_jina_ep2_oof.npy"):
    base = [PY, "-u", "jina/jina_ft.py", "--data", DATA, "--out", JF, "--folds", f"{ART}/folds.npy", "--fp16",
            "--epochs", 2, "--lr", "2e-5", "--warmup", 0.1, "--train_len", 512, "--infer_len", 1024, "--q_per_step", 8, "--max_docs", 16,
            "--bce_w", 0.5, "--seed", 42]
    split = [[0, 2, 4], [1, 3]] if NGPU >= 2 else [[0, 1, 2, 3, 4]]
    par([base + ["--run_folds", *s] for s in split], [f"{JF}/ft_gpu{k}.log" for k in range(len(split))]); sh(base + ["--merge"])

NameError: name 'NS' is not defined

## 8. Jina reranker v3 — zero-shot

In [ ]:
J3 = f"{ART}/jina_v3"; os.makedirs(J3, exist_ok=True)
if RUN["jina_v3_zs"] and not done(f"{J3}/zs_jv3_train.npy"):
    base = [PY, "-u", "jina/jina_v3_zs.py", "--data", DATA, "--out", J3, "--model", "jinaai/jina-reranker-v3", "--name", "zs_jv3",
            "--fp16", "--n_shards", NS, "--folds", f"{ART}/folds.npy"]
    par([base + ["--shard", k] for k in range(NS)], [f"{J3}/zs_s{k}.log" for k in range(NS)]); sh(base + ["--merge"])

## 9. Jina reranker v3 + LoRA — дообучение


In [ ]:
FV = f"{ART}/jina_v3_ft"; os.makedirs(FV, exist_ok=True)
if RUN["jina_v3_ft"] and not done(f"{FV}/ft_jv3_oof.npy"):
    base = [PY, "-u", "jina/jina_v3_ft.py", "--data", DATA, "--out", FV, "--folds", f"{ART}/folds.npy", "--model", "jinaai/jina-reranker-v3",
            "--fp16", "--train_minutes", 70]
    if NGPU >= 2:
        par([base + ["--run_folds", 0, 2, 4], base + ["--run_folds", 1, 3, "--test_folds", 1, 3]], [f"{FV}/v3ft_gpu0.log", f"{FV}/v3ft_gpu1.log"])
    else:
        sh(base + ["--run_folds", 0, 1, 2, 3, 4, "--test_folds", 1, 3], log=f"{FV}/v3ft_gpu0.log")
    sh(base + ["--merge"])

## 10. Финалка


In [ ]:
import importlib, src.metric as metric; importlib.reload(metric)
tr = pd.read_csv(f"{DATA}/train.csv"); te = pd.read_csv(f"{DATA}/test.csv"); ss = pd.read_csv(f"{DATA}/sample_submission.csv")
qc = pd.factorize(tr["query_id"].values)[0]; qt = pd.factorize(te["query_id"].values)[0]; folds = np.load(f"{ART}/folds.npy")

def zq(x, q):

    x = np.asarray(x, dtype=float); n = np.bincount(q)
    m = np.bincount(q, weights=x) / n; v = np.bincount(q, weights=x * x) / n - m * m
    return (x - m[q]) / (np.sqrt(np.maximum(v, 0)) + 1e-9)[q]

TRAIN = {"stack": f"{ART}/stack_bge5_qwen4b_zs_nomix_oof.npy", "jft2": f"{JF}/ft_jina_ep2_oof.npy",
         "jv3": f"{J3}/zs_jv3_train.npy", "fv3": f"{FV}/ft_jv3_oof.npy"}
TEST = {"stack": f"{ART}/stack_bge5_qwen4b_zs_nomix_test.npy", "jft2": f"{JF}/ft_jina_ep2_test.npy",
        "jv3": f"{J3}/zs_jv3_test.npy", "fv3": f"{FV}/ft_jv3_test.npy",
        "l0": f"{ART}/ce_qwen4b_lora_t4/test.npy", "l1": f"{ART}/ce_qwen4b_lora_t4_f1/test.npy"}
W = {"stack": 1.0, "jft2": 1.0, "jv3": 0.5, "fv3": 1.0, "lora": 0.25}

Ztr = {k: zq(np.load(p), qc) for k, p in TRAIN.items()}
for k in Ztr: print(f"OOF {k:5s}", round(metric.score(tr, Ztr[k]), 4))
oof = sum(W[k] * Ztr[k] for k in Ztr)
print("OOF бленд без lora:", round(metric.score(tr, oof), 4), "| по фолдам:", np.round(metric.fold_scores(tr, oof, folds), 4))

Zt = {k: zq(np.load(p), qt) for k, p in TEST.items()}
for k, v in Zt.items(): assert v.shape == (len(te),) and np.isfinite(v).all(), k
Zt["lora"] = zq((Zt.pop("l0") + Zt.pop("l1")) / 2, qt)
pred = sum(W[k] * Zt[k] for k in W)
t = te[["id", "query_id"]].copy(); t["prediction"] = pred
if t.duplicated(["query_id", "prediction"]).any():
    t["prediction"] += np.random.RandomState(0).uniform(0, 1e-6, len(t))
sub = ss[["id"]].merge(t[["id", "prediction"]], on="id", how="left")
assert len(sub) == len(ss) and sub.prediction.notna().all()
sub.to_csv("submission.csv", index=False); print(sub.shape); sub.head()

In [2]:
from google.colab import files
from pathlib import Path

submission_path = Path("submission.csv")
assert submission_path.is_file(), "Файл submission.csv не создан. Сначала выполни раздел 10."
files.download(str(submission_path))


AssertionError: Файл submission.csv не создан. Сначала выполни раздел 10.